# Diagnóstico de calidad de datos

**Proyecto:** Percepción de la belleza e influencia de las redes sociales
**Aprendiz:** Maileth Begambre · Tecnólogo en Análisis y Desarrollo de Software (ADSO)
**Evidencia:** GA1-220501094-01-AA1-EV06 – Diagnosticar la calidad de datos: preparación, exploración y documentación

Este cuaderno sigue la metodología de la Guía 4 (*Explorando Instacart*): **código → salida → interpretación**.
Las interpretaciones se escribieron a partir de las salidas reales de cada celda. No se usan datos de Instacart.

| Sección | Contenido | Momento EV06 |
|---|---|---|
| 1 | Comprensión del proyecto y de los datasets | 1 |
| 2 | Carga de los datos y diccionarios de códigos | 2 |
| 3 | Inspección de la estructura de cada dataset | 3 |
| 4 | Tabla de diagnóstico estructural | 3 |
| 5 | Diagnóstico de calidad: faltantes, tipos, inconsistencias, limitaciones, tabla de calidad y acciones | 4 |

## 1. Comprensión del proyecto y de los datasets

### Pregunta analítica (EV02)

> ¿Qué relación existe entre la frecuencia de exposición a contenido de belleza en redes sociales, la comparación de la apariencia con otras personas y los cambios que reportan los jóvenes de 18 a 25 años en su percepción de la belleza y sus preferencias de moda o maquillaje?

**Objetivo general.** Analizar, de forma descriptiva y exploratoria, la relación entre la exposición a contenido de belleza en redes sociales, la comparación de la apariencia y los cambios que reportan jóvenes de 18 a 25 años en su percepción de la belleza y sus preferencias de moda o maquillaje.

El proyecto separa tres dimensiones: **(1) exposición** (cuánto y qué contenido se ve), **(2) comparación** (si la persona compara su apariencia) y **(3) resultado percibido** (cambios en gustos, ideal de belleza o satisfacción). El análisis es descriptivo y exploratorio: no se afirmará causalidad.

### Datasets del proyecto

| DataFrame | Fuente | Formato | Qué contiene | Relación con el problema |
|---|---|---|---|---|
| `df_pritchard` | Pritchard (2021), Mendeley Data, CC BY 4.0 | SPSS `.sav` | Encuesta a estudiantes universitarios de EE. UU. sobre uso de Instagram, Twitter y Snapchat, comparación de apariencia e imagen corporal | Cubre exposición, comparación y satisfacción corporal. No mide moda ni maquillaje |
| `df_engeln` | Engeln et al. (2020), Mendeley Data, CC BY 4.0 | SPSS `.sav` | Experimento con mujeres universitarias de EE. UU.: 7 min de Instagram, Facebook o un juego de control; medidas antes y después | Cubre exposición controlada, comparación durante la tarea y satisfacción con la apariencia |
| `df_dane` | DANE, ENTIC Hogares 2024 (complementario, no se publica en GitHub) | CSV `;` | Encuesta oficial de uso de TIC en hogares de Colombia | Solo contexto colombiano de exposición: uso de internet y de redes sociales |

La **encuesta propia** es la fuente principal prevista, pero todavía no se ha aplicado; por eso no aparece en este cuaderno. Su diseño está en `data/diccionario_datos.md`.

### Aspectos de calidad que se deben revisar

- **Valores faltantes**, incluidos los que no aparecen como nulos (por ejemplo, textos vacíos) y los que se deben a preguntas que solo respondía una parte de las personas.
- **Tipos de datos:** las respuestas están guardadas como **códigos** (1, 2, 3…). Son categorías u órdenes, **no cantidades**: no se calcularán promedios de esos códigos sin justificarlo.
- **Rangos válidos:** edades dentro de 18 a 25 años y códigos que existan en el diccionario de etiquetas.
- **Identificadores y duplicados:** qué representa cada fila (persona, hogar, participante).
- **Cobertura:** población, país y variables que no existen en cada dataset.

### ¿Por qué explorar antes de analizar?

Porque un número en una tabla puede significar cosas distintas: una edad, un código de respuesta o un valor vacío. Si se analiza sin revisar la estructura, se pueden promediar categorías, contar filas como si fueran personas o ignorar respuestas faltantes, y las conclusiones serían incorrectas.

## 2. Preparación y carga de los datos

Cada dataset se carga **con el procedimiento que corresponde a su formato** y en un DataFrame independiente:

| Formato | Función | Por qué |
|---|---|---|
| SPSS `.sav` | `pd.read_spss(ruta, convert_categoricals=False)` | Conserva los **códigos originales** (1, 2, 3…) en lugar de reemplazarlos por texto |
| SPSS `.sav` (metadatos) | `pyreadstat.read_sav(ruta, metadataonly=True)` | Lee el diccionario que relaciona cada código con su etiqueta, sin cargar los datos otra vez |
| CSV del DANE | `pd.read_csv(ruta, sep=";", low_memory=False)` | El archivo usa punto y coma. `low_memory=False` hace que pandas revise la columna completa antes de decidir su tipo |

A diferencia de la Guía 4, aquí `sep=";"` solo aplica al CSV del DANE: los `.sav` no tienen separador.

In [1]:
from pathlib import Path
import re

import pandas as pd
import pyreadstat
from IPython.display import display

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 60)

# Funciona tanto si Jupyter se abre en la raíz del proyecto como en notebooks/
RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATOS = RAIZ / "data" / "raw"

RUTA_PRITCHARD = DATOS / "pritchard_2021_social_media_body_image.sav"
RUTA_ENGELN = DATOS / "engeln_2020_instagram_facebook.sav"
RUTA_DANE = DATOS / "dane" / "BASE DEFINITIVA ENTIC HOGARES 2024.csv"
RUTA_DANE_DICC = DATOS / "dane" / "DSO-ENTIC_H-DDT-001.xlsx"

for ruta in [RUTA_PRITCHARD, RUTA_ENGELN, RUTA_DANE, RUTA_DANE_DICC]:
    print("Encontrado " if ruta.exists() else "NO ENCONTRADO", ruta.relative_to(RAIZ))

Encontrado  data\raw\pritchard_2021_social_media_body_image.sav
Encontrado  data\raw\engeln_2020_instagram_facebook.sav
Encontrado  data\raw\dane\BASE DEFINITIVA ENTIC HOGARES 2024.csv
Encontrado  data\raw\dane\DSO-ENTIC_H-DDT-001.xlsx


In [2]:
df_pritchard = pd.read_spss(RUTA_PRITCHARD, convert_categoricals=False)
_, meta_pritchard = pyreadstat.read_sav(RUTA_PRITCHARD, metadataonly=True)

print("Pritchard cargado correctamente:", df_pritchard.shape)

Pritchard cargado correctamente: (404, 246)


In [3]:
df_engeln = pd.read_spss(RUTA_ENGELN, convert_categoricals=False)
_, meta_engeln = pyreadstat.read_sav(RUTA_ENGELN, metadataonly=True)

print("Engeln cargado correctamente:", df_engeln.shape)

Engeln cargado correctamente: (308, 70)


In [4]:
# El DANE no se publica en GitHub. Si falta, ver data/raw/README.md
if RUTA_DANE.exists():
    df_dane = pd.read_csv(RUTA_DANE, sep=";", low_memory=False)
    print("DANE cargado correctamente:", df_dane.shape)
else:
    df_dane = None
    print("No se encontró el CSV del DANE. Las celdas del DANE se omitirán.")
    print("Para descargarlo, sigue las instrucciones de data/raw/README.md")

DANE cargado correctamente: (148700, 377)


**Interpretación.** Los tres archivos se cargan sin errores, cada uno en su propio DataFrame. El CSV del DANE se lee con la codificación por defecto (UTF-8). Los números que muestran estas celdas son filas y columnas; en la sección 3 se revisa qué representa cada fila.

### 2.1 Diccionarios de códigos

Los archivos `.sav` guardan cada respuesta como un código y, aparte, la etiqueta de ese código (por ejemplo, `1 = Never`, `2 = Rarely`). Para no perder el significado, se construye un **diccionario de códigos** por dataset con cuatro columnas: variable, etiqueta de la variable (la pregunta), código y etiqueta del valor.

Los diccionarios de Pritchard y Engeln se guardan en `data/diccionarios/` para consultarlos sin abrir SPSS. El del DANE se arma desde su archivo oficial (`DSO-ENTIC_H-DDT-001.xlsx`) y no se exporta, porque el DANE no declara una licencia abierta.

In [5]:
def diccionario_codigos(meta):
    """Devuelve un DataFrame con variable, pregunta, código y etiqueta a partir de los metadatos SPSS."""
    filas = []
    for variable, pregunta in zip(meta.column_names, meta.column_labels):
        valores = meta.variable_value_labels.get(variable, {})
        if not valores:
            filas.append({"variable": variable, "etiqueta_variable": pregunta or "",
                          "codigo": pd.NA, "etiqueta_valor": pd.NA})
        for codigo, etiqueta in valores.items():
            filas.append({"variable": variable, "etiqueta_variable": pregunta or "",
                          "codigo": codigo, "etiqueta_valor": etiqueta})
    return pd.DataFrame(filas)


def etiquetas(diccionario, variable):
    """Muestra los códigos y etiquetas de una variable."""
    return diccionario.loc[diccionario["variable"] == variable, ["codigo", "etiqueta_valor"]]


dic_pritchard = diccionario_codigos(meta_pritchard)
dic_engeln = diccionario_codigos(meta_engeln)

CARPETA_DIC = RAIZ / "data" / "diccionarios"
CARPETA_DIC.mkdir(exist_ok=True)
dic_pritchard.to_csv(CARPETA_DIC / "pritchard_2021_codigos.csv", index=False)
dic_engeln.to_csv(CARPETA_DIC / "engeln_2020_codigos.csv", index=False)

for nombre, meta in [("Pritchard", meta_pritchard), ("Engeln", meta_engeln)]:
    sin_pregunta = sum(1 for etiqueta in meta.column_labels if not etiqueta)
    print(f"{nombre}: {len(meta.column_names)} variables, "
          f"{len(meta.variable_value_labels)} con etiquetas de valor, "
          f"{sin_pregunta} sin texto de pregunta")

Pritchard: 246 variables, 200 con etiquetas de valor, 21 sin texto de pregunta
Engeln: 70 variables, 56 con etiquetas de valor, 6 sin texto de pregunta


In [6]:
print("Pritchard · Q17:", meta_pritchard.column_names_to_labels["Q17"])
display(etiquetas(dic_pritchard, "Q17"))

print("Engeln · Condition:", meta_engeln.column_names_to_labels["Condition"])
display(etiquetas(dic_engeln, "Condition"))

Pritchard · Q17: What is the average amount of time you spend on Instagram each day?


,codigo,etiqueta_valor
14,1.0,0-15 min
15,2.0,15-30 min
16,3.0,1-2 hr
17,4.0,2-3 hr
18,5.0,3-4 hr
19,6.0,4-5 hr
20,7.0,5-6 hr
21,8.0,6-7 hr
22,9.0,7-8 hr
23,10.0,8-9 hr


Engeln · Condition: Condition


,codigo,etiqueta_valor
0,1.0,Browsed Facebook
1,2.0,Browsed Instagram
2,3.0,Played Bejeweled


**Interpretación.** El diccionario muestra que, por ejemplo, en Pritchard `Q17 = 3` no significa "3 horas" sino el intervalo **"1-2 hr"**, y que en Engeln `Condition = 2` significa **"Browsed Instagram"**. Por eso estos códigos se tratan como **categorías**, no como cantidades.

In [7]:
def diccionario_dane(ruta):
    """Arma el diccionario código → etiqueta a partir de la columna 'Dominios' del archivo oficial del DANE."""
    hoja = pd.read_excel(ruta, sheet_name="Diseño base de Datos", header=3)
    hoja.columns = [str(c).strip().split("\n")[0] for c in hoja.columns]
    filas = []
    for _, fila in hoja.dropna(subset=["Nombre de la variable o la columna"]).iterrows():
        variable = str(fila["Nombre de la variable o la columna"]).strip()
        pregunta = fila["Descripción de la variable o la columna"]
        tipo = fila["Tipo de dato"]
        dominio = str(fila["Dominios"]).replace("_x000D_", "\n")
        pares = re.findall(r"(\d+)\s*\.\s*([^\n]+)", dominio)
        if not pares:
            filas.append({"variable": variable, "etiqueta_variable": pregunta, "tipo_dato": tipo,
                          "codigo": pd.NA, "etiqueta_valor": pd.NA})
        for codigo, etiqueta in pares:
            filas.append({"variable": variable, "etiqueta_variable": pregunta, "tipo_dato": tipo,
                          "codigo": int(codigo), "etiqueta_valor": etiqueta.strip()})
    return pd.DataFrame(filas)


if RUTA_DANE_DICC.exists():
    dic_dane = diccionario_dane(RUTA_DANE_DICC)
    print("Variables en el diccionario del DANE:", dic_dane["variable"].nunique())
    print("IV12R13C1:", dic_dane.loc[dic_dane["variable"] == "IV12R13C1", "etiqueta_variable"].iloc[0])
    display(etiquetas(dic_dane, "IV12R13C1"))
else:
    dic_dane = None
    print("No se encontró el diccionario del DANE (ver data/raw/README.md).")

Variables en el diccionario del DANE: 377
IV12R13C1: ¿Utilizó las redes sociales para entretenimiento (ej. juego, farándula, seguir influenciadores)?


,codigo,etiqueta_valor
517,1,Sí
518,2,No


**Interpretación.** El archivo de datos del DANE no trae etiquetas: solo códigos de columna como `iv12r13c1`. El diccionario oficial permite saber que esa columna es la pregunta *"¿Utilizó las redes sociales para entretenimiento (ej. juego, farándula, seguir influenciadores)?"*, con `1 = Sí` y `2 = No`.

## 3. Inspección de la estructura de los datasets

Para cada DataFrame se ejecutan los comandos que pide la guía: `head()`, `tail()`, `shape`, `columns`, `dtypes`, `info()`, `describe()` y `describe(include="object")`.

**Advertencia para leer `describe()`:** calcula `mean` y `std` de todas las columnas numéricas, pero en estos datasets casi todas son **códigos de respuesta**. Esos promedios no tienen significado (el promedio de "Hombre = 1" y "Mujer = 2" no describe a nadie). De `describe()` se usan sobre todo `count` (cuántos valores no nulos hay), `min` y `max` (si los códigos están en el rango esperado).

### 3.1 Pritchard (2021) · `df_pritchard`

In [8]:
df_pritchard.head()

,age,gender,race,Q16,Q17,Q18,Q19,Q20,Q21,Q22,Q23,Q24,Q25,Q26,Q27,...,Eat_26,Dieting,bulimia,oralcontrol,filter_$,PC,PA,UBS,SA,WC,BodySurv,BodySham,Control,PACSR,GenderRE
0,18.0,2.0,5.0,1.0,3.0,3.0,2.0,,3.0,"Sometimes selfies, sometimes photos of me and my boyfrie...",As long as I don't look weird or am at an unflattering a...,2.0,,,2.0,...,5.0,5.0,0.0,0.0,1.0,22.0,39.0,20.0,42.0,28.0,3.375,1.625,3.500000,1.818182,0.0
1,17.0,2.0,3.0,1.0,2.0,2.0,1.0,I favor this time over others because it is most likely ...,2.0,I post pictures of myself with friends or with family. I...,"My first step would be to do my hair, my makeup, and put...",1.0,I use filters for the aesthetic of my feed.,I do consider how other users will respond to my images ...,2.0,...,2.0,1.0,0.0,1.0,1.0,32.0,43.0,29.0,48.0,38.0,3.000,2.000,3.500000,1.454545,0.0
2,18.0,2.0,5.0,1.0,3.0,4.0,1.0,I feel as though more people will be active.,3.0,I post selfies and pictures with my friends. I decide w...,I would send it to my friends to get input.,1.0,I feel like it makes me look better,I do because if I dont think I will get a lot of likes o...,1.0,...,29.0,22.0,2.0,5.0,1.0,30.0,27.0,26.0,32.0,28.0,4.375,3.250,3.714286,3.272727,0.0
3,18.0,1.0,5.0,1.0,2.0,3.0,1.0,Golden Hour,2.0,Pictures with friends or girlfriends. Ones that look goo...,Golden hour and non-blurry,2.0,,,2.0,...,5.0,1.0,1.0,3.0,0.0,37.0,35.0,28.0,44.0,31.0,3.000,1.875,3.625000,2.272727,1.0
4,19.0,2.0,5.0,1.0,1.0,2.0,2.0,,2.0,"I never post images of just myself, there is always at l...","I would make sure I have makeup on, and try to be in a p...",1.0,I edit and use filters to hide blemishes and make myself...,"Yes, I always make sure the picture is appropriate and t...",2.0,...,0.0,0.0,0.0,0.0,1.0,28.0,32.0,23.0,38.0,28.0,3.000,3.000,3.250000,2.545455,0.0


**Interpretación.** Cada fila es una persona que respondió la encuesta. Las primeras columnas son `age`, `gender` y `race`, seguidas de preguntas con nombres en código (`Q16`, `Q17`…). Los valores aparecen como `18.0`, `2.0`, `5.0`: son números con decimal aunque representen edades o códigos. En la columna de texto `Q20` la primera fila aparece **vacía**.

In [9]:
df_pritchard.tail()

,age,gender,race,Q16,Q17,Q18,Q19,Q20,Q21,Q22,Q23,Q24,Q25,Q26,Q27,...,Eat_26,Dieting,bulimia,oralcontrol,filter_$,PC,PA,UBS,SA,WC,BodySurv,BodySham,Control,PACSR,GenderRE
399,18.0,2.0,3.0,1.0,6.0,4.0,1.0,Most people go on instagram at night so that is when you...,4.0,Selfies or pictures taken of me. I decide by asking peop...,If it is relevant enough so that people will want to see...,1.0,To make it look better or to enhance the picture,"Yes, because that was the main purpose of posting the pi...",1.0,...,29.0,22.0,5.0,2.0,1.0,29.0,35.0,28.0,40.0,31.0,3.625,2.750,3.000000,1.454545,0.0
400,22.0,1.0,5.0,1.0,3.0,3.0,1.0,There's more activity on Instagram in the evening. Every...,2.0,I post photos of myself with friends or doing activities...,Make sure I look nice and am dressed appropriately.,2.0,,,2.0,...,6.0,6.0,0.0,0.0,0.0,43.0,45.0,36.0,58.0,37.0,3.500,2.625,3.714286,0.636364,1.0
401,18.0,2.0,5.0,1.0,3.0,4.0,1.0,Its relaxing,3.0,I mostly post selfies or pictures of activities im doing.,I would have to put on makeup and do my hair.,1.0,For better quality pictures,no I don't consider how others will respond.,2.0,...,3.0,3.0,0.0,0.0,1.0,27.0,33.0,27.0,41.0,31.0,3.000,NaN,5.000000,0.000000,0.0
402,18.0,1.0,1.0,1.0,2.0,5.0,1.0,"If pictures are posted at certain times of the day, more...",3.0,Used to post selfies but I feel like that's more for Sna...,Probably set the phone down or put the phone at an angle...,1.0,If i edit my pictures depending on what I want to get ac...,"Yes. Just because at this point, when I post a picture o...",2.0,...,3.0,0.0,0.0,3.0,0.0,29.0,34.0,18.0,41.0,27.0,3.500,NaN,5.000000,0.454545,1.0
403,18.0,1.0,5.0,1.0,3.0,5.0,2.0,,2.0,I don't really post a type. If I decide that it can be p...,"Would my friends, parents, others approve? Does it portr...",1.0,It enhances the photo to make it look better.,"Yes, I think that others will judge you by the photos yo...",2.0,...,5.0,5.0,0.0,0.0,0.0,35.0,35.0,28.0,42.0,29.0,3.125,2.875,3.250000,2.272727,1.0


**Interpretación.** Las últimas filas tienen la misma estructura; la numeración del índice termina en 403, porque empieza en 0. También aparecen celdas de texto vacías en las respuestas abiertas.

In [10]:
df_pritchard.shape

(404, 246)

**Interpretación.** El DataFrame tiene **404 filas y 246 columnas**. 404 es la cantidad de **registros**, no necesariamente de personas válidas para el proyecto: falta comprobar edades, duplicados y respuestas incompletas (sección 5).

In [11]:
print(df_pritchard.columns.tolist())

['age', 'gender', 'race', 'Q16', 'Q17', 'Q18', 'Q19', 'Q20', 'Q21', 'Q22', 'Q23', 'Q24', 'Q25', 'Q26', 'Q27', 'Q28', 'Q29', 'Q30', 'Q13_1', 'Q13_2', 'Q13_3', 'Q13_4', 'Q13_5', 'Q13_6', 'Q13_7', 'Q13_8', 'Q13_9', 'Q13_10', 'Q13_11', 'Q13_12', 'Q13_13', 'Q13_14', 'Q13_15', 'Q23_1', 'Q23_2', 'Q23_3', 'Q23_4', 'Q23_5', 'Q23_6', 'Q23_7', 'Q15_1', 'Q15_2', 'Q15_3', 'Q15_4', 'Q15_5', 'Q15_6', 'Q15_7', 'Q15_8', 'Q15_9', 'Q15_10', 'Q15_11', 'Q15_12', 'Q15_13', 'Q15_14', 'Q15_15', 'Q15_16', 'Q15_17', 'Q15_18', 'Q15_19', 'Q15_20', 'Q15_21', 'Q15_22', 'Q15_1.0', 'Q15_2.0', 'Q15_3.0', 'Q15_4.0', 'Q15_5.0', 'Q15_6.0', 'Q15_7.0', 'Q15_8.0', 'Q15_9.0', 'Q15_10.0', 'Q15_11.0', 'Q15_12.0', 'Q15_13.0', 'Q15_14.0', 'Q15_15.0', 'Q15_16.0', 'Q15_17.0', 'Q15_18.0', 'Q15_19.0', 'Q15_20.0', 'Q15_21.0', 'Q15_22.0', 'Q15_23', 'Q15_24', 'Q15_25', 'Q15_26', 'Q17_1', 'Q17_2', 'Q17_3', 'Q17_4', 'Q19_1', 'Q19_2', 'Q32', 'Q34', 'Q35', 'Q33', 'Q36', 'Q38', 'Q39', 'Q40', 'Q41', 'Q42', 'Q43', 'Q44', 'Q45', 'Q46', 'Q47', 

**Interpretación.** Los nombres son códigos del formulario original (`Q16`, `Q65_1`…), así que no se entienden sin el diccionario. Hay detalles que pueden confundir:
- La numeración no sigue el orden del cuestionario (`Q13_…` aparece después de `Q30`).
- `Q23` es una pregunta abierta y `Q23_1` … `Q23_7` son otra escala distinta.
- Hay nombres como `Q15_1` y `Q15_1.0`, que son preguntas diferentes.
- Al final hay puntajes ya calculados (`DFM`, `PACSR`, `BodySurv`…) y una variable técnica de SPSS (`filter_$`).

In [12]:
df_pritchard.dtypes

age         float64
gender      float64
race        float64
Q16         float64
Q17         float64
             ...   
BodySurv    float64
BodySham    float64
Control     float64
PACSR       float64
GenderRE    float64
Length: 246, dtype: object

In [13]:
df_pritchard.dtypes.value_counts()

float64    221
str         25
Name: count, dtype: int64

**Interpretación.** Hay **221 columnas `float64`** (números con decimal) y **25 columnas de texto** (`str`; en pandas 3 el texto ya no aparece como `object`). Las respuestas cerradas (`gender`, `Q17`…) están guardadas como `float64`, pero son **códigos de categorías**. El tipo no indica que sean cantidades. La única variable claramente cuantitativa al inicio es `age`.

In [14]:
df_pritchard.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 404 entries, 0 to 403
Data columns (total 246 columns):
 #    Column          Non-Null Count  Dtype  
---   ------          --------------  -----  
 0    age             403 non-null    float64
 1    gender          403 non-null    float64
 2    race            403 non-null    float64
 3    Q16             404 non-null    float64
 4    Q17             385 non-null    float64
 5    Q18             371 non-null    float64
 6    Q19             384 non-null    float64
 7    Q20             404 non-null    str    
 8    Q21             385 non-null    float64
 9    Q22             404 non-null    str    
 10   Q23             404 non-null    str    
 11   Q24             361 non-null    float64
 12   Q25             404 non-null    str    
 13   Q26             404 non-null    str    
 14   Q27             362 non-null    float64
 15   Q28             404 non-null    str    
 16   Q29             404 non-null    str    
 17   Q30             404 non-n

**Interpretación.** `info()` muestra, para cada columna, cuántos valores **no nulos** tiene. Muchas columnas tienen menos de 404; las que menos tienen son `Q41` y `Q44` (54 valores), que pertenecen al bloque de preguntas sobre **Twitter**. Una explicación probable es que esas preguntas solo las respondían quienes usan Twitter; se verificará en la sección 5.

Las columnas de texto aparecen con **404 no nulos**, pero en `head()` ya se vieron celdas vacías. Eso significa que las respuestas en blanco están guardadas como texto vacío (`""`) y **no se cuentan como faltantes**.

In [15]:
df_pritchard.describe()

,age,gender,race,Q16,Q17,Q18,Q19,Q21,Q24,Q27,Q13_1,Q13_2,Q13_3,Q13_4,Q13_5,...,Eat_26,Dieting,bulimia,oralcontrol,filter_$,PC,PA,UBS,SA,WC,BodySurv,BodySham,Control,PACSR,GenderRE
count,403.000000,403.000000,403.000000,404.000000,385.000000,371.000000,384.000000,385.000000,361.000000,362.000000,402.000000,403.000000,402.000000,402.000000,403.000000,...,402.000000,402.000000,402.000000,402.000000,403.000000,403.000000,403.000000,403.000000,403.000000,403.000000,391.000000,359.000000,389.000000,395.000000,398.000000
mean,18.627792,1.615385,4.838710,1.047030,3.023377,4.013477,1.552083,2.636364,1.318560,1.657459,4.315920,3.297767,2.385572,2.159204,1.950372,...,10.544776,6.671642,1.405473,2.467662,0.583127,31.069479,37.506203,26.560794,45.168734,30.933002,3.347278,2.654596,3.707951,1.785316,0.409548
std,1.367869,0.540372,0.881705,0.211965,1.366250,1.530706,0.497929,0.772393,0.466564,0.475216,1.361089,1.614816,1.605194,1.543881,1.390271,...,12.038352,7.773389,2.898494,3.292885,0.493654,7.856938,7.630903,6.569805,8.674106,9.410113,0.756784,0.843703,0.637097,1.209592,0.492369
min,17.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,11.000000,8.000000,13.000000,10.000000,1.000000,1.000000,1.250000,0.000000,0.000000
25%,18.000000,1.000000,5.000000,1.000000,2.000000,3.000000,1.000000,2.000000,1.000000,1.000000,3.000000,2.000000,1.000000,1.000000,1.000000,...,3.000000,1.250000,0.000000,0.000000,0.000000,26.000000,33.000000,23.000000,39.500000,25.000000,2.875000,2.000000,3.250000,0.909091,0.000000
50%,18.000000,2.000000,5.000000,1.000000,3.000000,4.000000,2.000000,3.000000,1.000000,2.000000,4.000000,3.000000,2.000000,1.000000,1.000000,...,6.000000,4.000000,0.000000,2.000000,1.000000,32.000000,36.000000,26.000000,45.000000,31.000000,3.375000,2.625000,3.750000,1.818182,0.000000
75%,19.000000,2.000000,5.000000,1.000000,3.000000,5.000000,2.000000,3.000000,2.000000,2.000000,5.750000,5.000000,3.000000,3.000000,2.000000,...,13.000000,9.000000,1.000000,3.000000,1.000000,37.000000,43.000000,31.000000,51.000000,38.000000,3.875000,3.250000,4.125000,2.727273,1.000000
max,26.000000,4.000000,7.000000,2.000000,12.000000,7.000000,2.000000,4.000000,2.000000,2.000000,6.000000,6.000000,6.000000,6.000000,6.000000,...,72.000000,39.000000,18.000000,21.000000,1.000000,45.000000,55.000000,40.000000,65.000000,50.000000,4.875000,5.000000,5.000000,4.000000,1.000000


**Interpretación.**
- `age`: `count` = 403, así que falta una edad. `min` = 17 y `max` = 26: hay edades **fuera del rango de 18 a 25** del proyecto.
- `gender`: va de 1 a 4. Según el diccionario, son categorías (Male, Female, Transgender, Other); su `mean` no se interpreta.
- `Q17` (tiempo diario en Instagram): va de 1 a 12. Son **intervalos de tiempo codificados**, no horas.
- Las variables tienen `count` distintos, lo que confirma que la cantidad de respuestas cambia de una pregunta a otra.

In [16]:
df_pritchard.describe(include="object")

C:\Users\maile\AppData\Local\Temp\ipykernel_5308\3351535375.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_pritchard.describe(include="object")


,Q20,Q22,Q23,Q25,Q26,Q28,Q29,Q30,Q36,Q39,Q40,Q42,Q43,Q45,Q46,Q47,Q53,Q55,Q56,Q58,Q59,Q61,Q62,Q63,id
count,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404,404
unique,170,361,354,243,244,78,121,381,7,55,55,25,25,6,8,173,16,283,277,147,146,22,34,329,403
top,,,,,,,,,,,,,,,,,,,,,,,,,10720
freq,234,44,46,159,159,282,282,24,398,350,350,380,380,397,397,223,389,120,121,255,255,367,367,62,2


**Interpretación.** Las 25 columnas de texto tienen `count` = 404, pero el valor más frecuente (`top`) que aparece en blanco es el **texto vacío**. Por ejemplo, en `Q20` el texto vacío se repite 234 veces (`freq`). Esto confirma que buena parte de las respuestas abiertas no se contestó, aunque pandas no lo cuente como nulo.

**Sobre el aviso `Pandas4Warning`.** En pandas 3 el texto tiene su propio tipo (`str`). `include="object"` todavía incluye esas columnas por compatibilidad, y por eso el resultado es correcto, pero pandas avisa que en una versión futura dejará de hacerlo. Se conserva el comando porque es el que pide la guía; en pandas 4 habría que escribir `describe(include="str")`.

### 3.2 Engeln et al. (2020) · `df_engeln`

In [17]:
df_engeln.head()

,Condition,NA1,PA1,NA2,PA2,NA3,PA3,NA4,PA4,PA5,NA5,NA6,PA6,NA7,PA7,...,NegAffectPost,PosAffectPost,BISSTotalPre,BISSTotalPost,imagepeople,imageobject,reading,videopeople,videoobject,other,appearancethoughts,overallappcompduringtask,specbodypartcompduringtask,comparisonsduringtask,Age
0,1.0,2.0,4.0,4.0,3.0,2.0,3.0,3.0,5.0,4.0,1.0,1.0,3.0,1.0,4.0,...,1.9,3.9,7.166667,7.333333,0.0,15.0,25.0,0.0,60.0,0.0,1.0,1.0,1.0,1.0,19.0
1,1.0,1.0,1.0,3.0,1.0,1.0,1.0,2.0,2.0,1.0,1.0,1.0,1.0,2.0,NaN,...,2.6,1.6,3.666667,3.833333,0.0,50.0,20.0,30.0,0.0,0.0,1.0,1.0,1.0,1.0,19.0
2,1.0,1.0,3.0,1.0,4.0,1.0,3.0,3.0,4.0,4.0,1.0,2.0,4.0,1.0,3.0,...,1.7,2.8,6.666667,4.166667,0.0,45.0,5.0,50.0,0.0,0.0,1.0,1.0,1.0,1.0,19.0
3,1.0,1.0,3.0,1.0,4.0,1.0,1.0,1.0,3.0,2.0,1.0,1.0,3.0,2.0,4.0,...,1.4,2.8,5.833333,5.500000,5.0,50.0,35.0,5.0,5.0,0.0,1.0,1.0,1.0,1.0,18.0
4,1.0,1.0,1.0,4.0,3.0,1.0,3.0,2.0,3.0,3.0,1.0,1.0,4.0,3.0,4.0,...,1.2,3.2,6.000000,6.000000,0.0,45.0,0.0,5.0,50.0,0.0,2.0,1.0,1.0,1.0,20.0


**Interpretación.** Cada fila es una participante del experimento. La primera columna, `Condition`, indica el grupo (en las primeras filas es 1, *Browsed Facebook*). Siguen ítems de emociones con nombres abreviados (`NA1` = emoción negativa 1, `PA1` = emoción positiva 1), todos como números con decimal.

In [18]:
df_engeln.tail()

,Condition,NA1,PA1,NA2,PA2,NA3,PA3,NA4,PA4,PA5,NA5,NA6,PA6,NA7,PA7,...,NegAffectPost,PosAffectPost,BISSTotalPre,BISSTotalPost,imagepeople,imageobject,reading,videopeople,videoobject,other,appearancethoughts,overallappcompduringtask,specbodypartcompduringtask,comparisonsduringtask,Age
303,3.0,1.0,2.0,1.0,2.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.1,2.1,3.000000,3.666667,NaN,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,18.0
304,3.0,1.0,5.0,1.0,5.0,1.0,3.0,3.0,3.0,3.0,1.0,1.0,4.0,1.0,4.0,...,1.3,3.9,5.166667,5.166667,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,20.0
305,3.0,1.0,2.0,1.0,3.0,1.0,3.0,2.0,2.0,2.0,1.0,1.0,2.0,3.0,2.0,...,1.8,2.6,4.166667,4.166667,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN,NaN,NaN,18.0
306,3.0,1.0,2.0,3.0,4.0,1.0,1.0,2.0,2.0,1.0,1.0,1.0,3.0,3.0,2.0,...,2.4,2.7,4.333333,4.500000,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,19.0
307,3.0,1.0,3.0,1.0,3.0,1.0,1.0,1.0,3.0,1.0,1.0,1.0,1.0,1.0,1.0,...,1.1,2.5,6.166667,7.000000,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,19.0


**Interpretación.** Las últimas filas pertenecen a `Condition` = 3 (*Played Bejeweled*, el grupo de control). El archivo parece estar **ordenado por grupo**.

In [19]:
df_engeln.shape

(308, 70)

**Interpretación.** Tiene **308 filas y 70 columnas**. Es mucho más pequeño que Pritchard y está centrado en un experimento puntual.

In [20]:
print(df_engeln.columns.tolist())

['Condition', 'NA1', 'PA1', 'NA2', 'PA2', 'NA3', 'PA3', 'NA4', 'PA4', 'PA5', 'NA5', 'NA6', 'PA6', 'NA7', 'PA7', 'NA8', 'PA8', 'PA9', 'NA9', 'PA10', 'NA10', 'BISS1pre', 'BISS2pre', 'BISS3pre', 'BISS4pre', 'BISS5pre', 'BISS6pre', 'NA1post', 'PA1post', 'NA2post', 'PA2post', 'NA3post', 'PA3post', 'NA4post', 'PA4post', 'PA5post', 'NA5post', 'NA6post', 'PA6post', 'NA7post', 'PA7post', 'NA8post', 'PA8post', 'PA9post', 'NA9post', 'PA10post', 'NA10post', 'BISS1post', 'BISS2post', 'BISS3post', 'BISS4post', 'BISS5post', 'BISS6post', 'NegAffectPre', 'PosAffectPre', 'NegAffectPost', 'PosAffectPost', 'BISSTotalPre', 'BISSTotalPost', 'imagepeople', 'imageobject', 'reading', 'videopeople', 'videoobject', 'other', 'appearancethoughts', 'overallappcompduringtask', 'specbodypartcompduringtask', 'comparisonsduringtask', 'Age']


**Interpretación.** Las columnas siguen el orden del experimento: emociones **antes** (`NA1`…`PA10`), satisfacción corporal antes (`BISS1pre`…), emociones y satisfacción **después** (`…post`), puntajes totales (`BISSTotalPre`, `BISSTotalPost`), porcentaje de tiempo mirando cada tipo de contenido (`imagepeople`…), comparación durante la tarea y `Age`. No hay variable de género porque todas las participantes eran mujeres.

In [21]:
df_engeln.dtypes

Condition                     float64
NA1                           float64
PA1                           float64
NA2                           float64
PA2                           float64
                               ...   
appearancethoughts            float64
overallappcompduringtask      float64
specbodypartcompduringtask    float64
comparisonsduringtask         float64
Age                           float64
Length: 70, dtype: object

In [22]:
df_engeln.dtypes.value_counts()

float64    70
Name: count, dtype: int64

**Interpretación.** Las **70 columnas son `float64`**. `Condition` es una categoría (1, 2, 3) aunque esté guardada como número. Los ítems de emociones y de comparación son escalas ordinales. Algunas columnas sí son cantidades: `Age`, los porcentajes de tiempo y los puntajes totales.

In [23]:
df_engeln.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 308 entries, 0 to 307
Data columns (total 70 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Condition                   308 non-null    float64
 1   NA1                         308 non-null    float64
 2   PA1                         308 non-null    float64
 3   NA2                         307 non-null    float64
 4   PA2                         308 non-null    float64
 5   NA3                         308 non-null    float64
 6   PA3                         308 non-null    float64
 7   NA4                         308 non-null    float64
 8   PA4                         308 non-null    float64
 9   PA5                         308 non-null    float64
 10  NA5                         308 non-null    float64
 11  NA6                         308 non-null    float64
 12  PA6                         307 non-null    float64
 13  NA7                         308 non-null    fl

**Interpretación.** La mayoría de columnas tiene 308 valores, pero las variables de la tarea (`imagepeople`, `reading`, `overallappcompduringtask`, `comparisonsduringtask`…) tienen **201 o 202**. Esas preguntas se refieren a lo que se vio en Instagram o Facebook, así que es probable que el grupo de control no las respondiera; se verificará en la sección 5. `BISSTotalPre` tiene 300, por lo que faltan algunos puntajes iniciales.

In [24]:
df_engeln.describe()

,Condition,NA1,PA1,NA2,PA2,NA3,PA3,NA4,PA4,PA5,NA5,NA6,PA6,NA7,PA7,...,NegAffectPost,PosAffectPost,BISSTotalPre,BISSTotalPost,imagepeople,imageobject,reading,videopeople,videoobject,other,appearancethoughts,overallappcompduringtask,specbodypartcompduringtask,comparisonsduringtask,Age
count,308.000000,308.000000,308.000000,307.000000,308.000000,308.000000,308.000000,308.000000,308.000000,308.000000,308.000000,308.000000,307.000000,308.000000,307.000000,...,308.000000,308.000000,300.000000,308.000000,202.000000,201.000000,202.000000,202.000000,201.000000,201.000000,308.000000,202.000000,202.000000,202.000000,308.000000
mean,2.022727,1.269481,2.418831,1.869707,3.084416,1.233766,1.970779,1.840909,2.873377,2.461039,1.279221,1.292208,2.364821,1.970779,2.716612,...,1.595779,2.488961,4.517778,4.484307,36.410891,26.706468,11.393564,12.094059,8.457711,5.136816,2.487013,3.331683,3.009901,3.170792,19.396104
std,0.816844,0.667127,1.093318,1.085839,0.923464,0.663543,1.015735,1.048039,0.909705,1.028158,0.712812,0.773547,1.077306,1.136795,1.120728,...,0.583666,0.849377,1.351829,1.337543,27.679482,22.474839,13.301136,16.994763,15.651819,13.799364,1.649151,1.804882,1.884712,1.705067,1.231608
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.100000,1.100000,0.833333,0.500000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000,18.000000
25%,1.000000,1.000000,2.000000,1.000000,3.000000,1.000000,1.000000,1.000000,2.000000,2.000000,1.000000,1.000000,1.000000,1.000000,2.000000,...,1.100000,1.800000,3.500000,3.666667,10.000000,10.000000,2.000000,0.000000,0.000000,0.000000,1.000000,2.000000,1.000000,1.500000,18.000000
50%,2.000000,1.000000,2.000000,2.000000,3.000000,1.000000,2.000000,1.500000,3.000000,2.000000,1.000000,1.000000,2.000000,2.000000,3.000000,...,1.400000,2.400000,4.333333,4.166667,30.000000,20.000000,10.000000,5.000000,0.000000,0.000000,2.000000,3.000000,2.000000,3.000000,19.000000
75%,3.000000,1.000000,3.000000,2.000000,4.000000,1.000000,3.000000,2.000000,3.000000,3.000000,1.000000,1.000000,3.000000,3.000000,3.500000,...,1.800000,3.000000,5.500000,5.500000,60.000000,40.000000,15.000000,18.750000,10.000000,3.000000,4.000000,5.000000,5.000000,4.500000,20.000000
max,3.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,...,5.000000,5.000000,8.000000,8.500000,90.000000,95.000000,85.000000,80.000000,100.000000,90.000000,7.000000,7.000000,7.000000,7.000000,26.000000


**Interpretación.**
- `Condition`: va de 1 a 3, coherente con los tres grupos del diccionario.
- `Age`: va de 18 a **26**, así que hay al menos una edad fuera del rango del proyecto.
- `comparisonsduringtask`: va de 1 a 7, como su escala de "no comparison" a "a lot of comparison"; `count` = 202 confirma los faltantes de la tarea.
- Los `mean` de ítems ordinales (`NA1`, `PA1`…) no se interpretan como cantidades.

In [25]:
columnas_texto = df_engeln.select_dtypes(include="object").columns
if len(columnas_texto) > 0:
    display(df_engeln.describe(include="object"))
else:
    print("df_engeln no tiene columnas de texto: describe(include='object') no aplica.")

df_engeln no tiene columnas de texto: describe(include='object') no aplica.


**Interpretación.** Engeln no tiene columnas de texto. Si se ejecuta `describe(include="object")` directamente, pandas da un error `ValueError`, porque no encuentra columnas de ese tipo. Por eso la celda primero comprueba si existen, como indica la guía: *"cuando existan"*.

### 3.3 DANE ENTIC Hogares 2024 · `df_dane`

Las celdas de esta sección solo se ejecutan si el CSV del DANE está en `data/raw/dane/`.

In [26]:
if df_dane is not None:
    display(df_dane.head())

,ID,UID_ENCUESTA,uid_padre,UUID,UUID_1,p1r1c1,p2r1c1,p3r1c1,FEX_c,ii1r1c1,ii1r1c2,ii2r1c1,ii3r1c1,ii4r1c1,ii5r1c1,...,v6r2c1,v6r3c1,v6r4c1,v6r5c1,v6r6c1,v6r7c1,v6r8c1,v6r9c1,v6r10c1,v7r1c1,v7r2c1,v7r3c1,v7r4c1,v7r5c1,v7r6c1
0,59694,a0dc1da3-ce83-4d87-860e-d451da72dbc4,a0dc1da3-ce83-4d87-860e-d451da72dbc4,a0dc1da3-ce83-4d87-860e-d451da72dbc4/Hogares[1],a0dc1da3-ce83-4d87-860e-d451da72dbc4/Hogares[1]/Personas[1],15,15001,1,"149,227493",1,3.0,1,3,3,2,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
1,59694,a0dc1da3-ce83-4d87-860e-d451da72dbc4,a0dc1da3-ce83-4d87-860e-d451da72dbc4,a0dc1da3-ce83-4d87-860e-d451da72dbc4/Hogares[1],a0dc1da3-ce83-4d87-860e-d451da72dbc4/Hogares[1]/Personas[2],15,15001,1,"149,227493",1,3.0,1,3,3,2,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
2,59694,a0dc1da3-ce83-4d87-860e-d451da72dbc4,a0dc1da3-ce83-4d87-860e-d451da72dbc4,a0dc1da3-ce83-4d87-860e-d451da72dbc4/Hogares[1],a0dc1da3-ce83-4d87-860e-d451da72dbc4/Hogares[1]/Personas[3],15,15001,1,"149,227493",1,3.0,1,3,3,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,59695,7a8612ad-6679-402f-9800-1b31be670a17,7a8612ad-6679-402f-9800-1b31be670a17,7a8612ad-6679-402f-9800-1b31be670a17/Hogares[1],7a8612ad-6679-402f-9800-1b31be670a17/Hogares[1]/Personas[1],15,15001,1,"231,7540904",1,3.0,1,4,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,59695,7a8612ad-6679-402f-9800-1b31be670a17,7a8612ad-6679-402f-9800-1b31be670a17,7a8612ad-6679-402f-9800-1b31be670a17/Hogares[1],7a8612ad-6679-402f-9800-1b31be670a17/Hogares[1]/Personas[2],15,15001,1,"231,7540904",1,3.0,1,4,1,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Interpretación.** Las primeras columnas son identificadores (`ID`, `UID_ENCUESTA`, `uid_padre`, `UUID`…), luego `p1r1c1` (departamento), `p2r1c1` (municipio) y `FEX_c` (factor de expansión). Las preguntas tienen nombres en código (`ii1r1c1`…). `FEX_c` aparece como `149,227493`: un número escrito **con coma decimal**.

In [27]:
if df_dane is not None:
    display(df_dane.tail())

,ID,UID_ENCUESTA,uid_padre,UUID,UUID_1,p1r1c1,p2r1c1,p3r1c1,FEX_c,ii1r1c1,ii1r1c2,ii2r1c1,ii3r1c1,ii4r1c1,ii5r1c1,...,v6r2c1,v6r3c1,v6r4c1,v6r5c1,v6r6c1,v6r7c1,v6r8c1,v6r9c1,v6r10c1,v7r1c1,v7r2c1,v7r3c1,v7r4c1,v7r5c1,v7r6c1
148695,119603,e465e8a7-bccd-42cd-a9ea-5562806fbcc0,e465e8a7-bccd-42cd-a9ea-5562806fbcc0,e465e8a7-bccd-42cd-a9ea-5562806fbcc0/Hogares[1],e465e8a7-bccd-42cd-a9ea-5562806fbcc0/Hogares[1]/Personas[1],76,76001,1,"1490,035081",1,2.0,1,1,2,2,...,2.0,3.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,1.0
148696,119604,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1],cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1]/Personas[1],76,76001,1,"2152,610599",1,1.0,1,4,3,1,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
148697,119604,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1],cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1]/Personas[2],76,76001,1,"2152,610599",1,1.0,1,4,3,1,...,3.0,3.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
148698,119604,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1],cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1]/Personas[3],76,76001,1,"2152,610599",1,1.0,1,4,3,1,...,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
148699,119604,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f,cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1],cd8a96be-925b-4186-b46c-668155e30e9f/Hogares[1]/Personas[4],76,76001,1,"2152,610599",1,1.0,1,4,3,1,...,2.0,2.0,2.0,2.0,2.0,2.0,1.0,1.0,2.0,1.0,2.0,2.0,1.0,NaN,NaN


**Interpretación.** Las últimas filas mantienen la estructura. Muchas columnas aparecen como `NaN`, lo que anticipa una gran cantidad de valores vacíos.

In [28]:
if df_dane is not None:
    print(df_dane.shape)
    print("Encuestas (UID_ENCUESTA) distintas:", df_dane["UID_ENCUESTA"].nunique())

(148700, 377)
Encuestas (UID_ENCUESTA) distintas: 53343


**Interpretación.** El archivo tiene **148.700 filas y 377 columnas**, pero solo **53.343 valores distintos de `UID_ENCUESTA`**. Igual que en la Guía 4, donde las filas no eran clientes, aquí **cada fila es una persona** y varias personas comparten la misma encuesta de hogar. El número de filas no es el número de hogares.

In [29]:
if df_dane is not None:
    print(df_dane.columns.tolist())

['ID', 'UID_ENCUESTA', 'uid_padre', 'UUID', 'UUID_1', 'p1r1c1', 'p2r1c1', 'p3r1c1', 'FEX_c', 'ii1r1c1', 'ii1r1c2', 'ii2r1c1', 'ii3r1c1', 'ii4r1c1', 'ii5r1c1', 'ii7r1c1', 'ii6r1c2', 'ii7r2c1', 'ii6r2c2', 'ii7r3c1', 'ii6r3c2', 'ii7r4c1', 'ii6r4c2', 'ii7r5c1', 'ii6r5c2', 'ii7r6c1', 'ii6r6c2', 'ii9r1c1', 'ii9r2c1', 'ii9r3c1', 'ii9r4c1', 'ii10r1c1', 'ii10r1c2', 'ii10r1c3', 'ii11r1c1', 'ii12r1c1', 'ii13r1c1', 'ii14r1c1', 'ii15ar1c1', 'ii15r1c1', 'ii17r1c1', 'ii18r1c1', 'ii18r1c2', 'ii18r1c3', 'ii18r2c1', 'ii18r3c1', 'ii18r4c1', 'ii8r1c1', 'ii16r1c1', 'ii19r1c1', 'ii19r2c1', 'ii19r3c1', 'ii19r4c1', 'ii19r5c1', 'iii2r1c1', 'iii4r1c1', 'iii5r1c1', 'iii6r1c1', 'iii7r1c1', 'iii8r1c1', 'iii10r1c1', 'iii11r1c1', 'iii11r2c1', 'iii11r3c1', 'iii11r4c1', 'iii11r5c1', 'iii11r6c1', 'iii11r7c1', 'iii11r8c1', 'iii12r1c1', 'iii13r1c1', 'iii14r1c1', 'iii15r1c1', 'iii16r1c1', 'iii16r1c2', 'iii16r1c3', 'iii17r1c1', 'iii18r1c1', 'iii19r1c1', 'iv1r1c1', 'iv2r1c1', 'iv3r1c1', 'iv4r1c1', 'iv4r2c1', 'iv4r3c1', 'iv4

**Interpretación.** Los nombres son códigos de pregunta (`iv12r13c1` = sección IV, pregunta 12, fila 13, columna 1). Están en **minúscula** (excepto `FEX_c` y algunos identificadores), mientras que el diccionario oficial los escribe en **mayúscula** (`IV12R13C1`). Para relacionarlos hay que comparar sin distinguir mayúsculas.

In [30]:
if df_dane is not None:
    display(df_dane.dtypes)

ID                int64
UID_ENCUESTA        str
uid_padre           str
UUID                str
UUID_1              str
                 ...   
v7r2c1          float64
v7r3c1          float64
v7r4c1          float64
v7r5c1          float64
v7r6c1          float64
Length: 377, dtype: object

In [31]:
if df_dane is not None:
    display(df_dane.dtypes.value_counts())

float64    347
int64       25
str          5
Name: count, dtype: int64

**Interpretación.** Hay **347 columnas `float64`, 25 `int64` y 5 de texto**. Las de texto son cuatro identificadores y `FEX_c`. Que `FEX_c` sea texto es un problema: es un **factor de expansión** (un número) y, por estar escrito con coma decimal, pandas no lo reconoce como número. Las columnas `float64` son, en su mayoría, códigos de respuesta (por ejemplo, `1 = Sí`, `2 = No`).

In [32]:
if df_dane is not None:
    df_dane.info(verbose=True, show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 148700 entries, 0 to 148699
Data columns (total 377 columns):
 #    Column        Non-Null Count   Dtype  
---   ------        --------------   -----  
 0    ID            148700 non-null  int64  
 1    UID_ENCUESTA  148700 non-null  str    
 2    uid_padre     148700 non-null  str    
 3    UUID          148700 non-null  str    
 4    UUID_1        148700 non-null  str    
 5    p1r1c1        148700 non-null  int64  
 6    p2r1c1        148700 non-null  int64  
 7    p3r1c1        148700 non-null  int64  
 8    FEX_c         148700 non-null  str    
 9    ii1r1c1       148700 non-null  int64  
 10   ii1r1c2       144655 non-null  float64
 11   ii2r1c1       148700 non-null  int64  
 12   ii3r1c1       148700 non-null  int64  
 13   ii4r1c1       148700 non-null  int64  
 14   ii5r1c1       148700 non-null  int64  
 15   ii7r1c1       148700 non-null  int64  
 16   ii6r1c2       40015 non-null   float64
 17   ii7r2c1       148700 non-null  int64  

**Interpretación.** Solo una parte de las columnas está completa en las 148.700 filas; la mayoría tiene faltantes y una (`iv13r20c2`) no tiene ningún valor. En una encuesta de hogares esto suele deberse a **preguntas que solo se hacen a algunas personas** (por ejemplo, las de redes sociales solo a quien usa internet para redes). El conteo exacto y su explicación se revisan en la sección 5.

In [33]:
if df_dane is not None:
    display(df_dane.describe())

,ID,p1r1c1,p2r1c1,p3r1c1,ii1r1c1,ii1r1c2,ii2r1c1,ii3r1c1,ii4r1c1,ii5r1c1,ii7r1c1,ii6r1c2,ii7r2c1,ii6r2c2,ii7r3c1,...,v6r2c1,v6r3c1,v6r4c1,v6r5c1,v6r6c1,v6r7c1,v6r8c1,v6r9c1,v6r10c1,v7r1c1,v7r2c1,v7r3c1,v7r4c1,v7r5c1,v7r6c1
count,148700.000000,148700.000000,148700.000000,148700.000000,148700.000000,144655.000000,148700.000000,148700.000000,148700.000000,148700.000000,148700.000000,40015.000000,148700.000000,89032.000000,148700.000000,...,99064.000000,99064.000000,99064.000000,99064.000000,99064.000000,99064.000000,99064.000000,99064.000000,99064.000000,6964.000000,6964.000000,4051.000000,6964.000000,3051.000000,3591.000000
mean,87446.210377,44.774264,45062.669274,1.612495,1.027202,2.028654,1.010282,3.562703,2.565293,1.918978,1.730901,1.077321,1.401264,1.300297,1.969153,...,2.019018,2.005411,2.002776,2.002140,2.007904,2.021309,2.002837,2.009024,2.013769,1.483199,1.747128,1.681560,1.483199,1.355293,1.797549
std,16417.993931,29.673332,29660.203245,0.861428,0.162673,1.771237,0.123153,1.684965,1.037962,0.272870,0.443493,0.363625,0.490156,0.681809,0.172905,...,0.168006,0.195213,0.183130,0.180669,0.155482,0.176385,0.189469,0.148479,0.133564,0.499754,0.434689,0.465929,0.499754,0.478681,0.401882
min,59694.000000,5.000000,5001.000000,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,73369.750000,17.000000,17867.000000,1.000000,1.000000,1.000000,1.000000,2.000000,2.000000,2.000000,1.000000,1.000000,1.000000,1.000000,2.000000,...,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000
50%,86773.000000,44.000000,44430.000000,1.000000,1.000000,2.000000,1.000000,3.000000,2.000000,2.000000,2.000000,1.000000,1.000000,1.000000,2.000000,...,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,1.000000,2.000000,2.000000,1.000000,1.000000,2.000000
75%,101060.000000,70.000000,70670.000000,3.000000,1.000000,2.000000,1.000000,4.000000,3.000000,2.000000,2.000000,1.000000,2.000000,1.000000,2.000000,...,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
max,119604.000000,99.000000,99773.000000,3.000000,2.000000,9.000000,4.000000,16.000000,7.000000,2.000000,2.000000,16.000000,2.000000,38.000000,2.000000,...,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000


**Interpretación.** `describe()` del DANE resume 372 columnas numéricas, casi todas códigos. Por eso se revisan aparte las variables pertinentes para el proyecto (siguiente celda).

In [34]:
if df_dane is not None:
    VARIABLES_DANE = {
        "p1r1c1": "Departamento (código)",
        "p2r1c1": "Municipio (código)",
        "iii4r1c1": "Sexo al nacer",
        "iii7r1c1": "Años cumplidos",
        "iv5r1c1": "Frecuencia de uso de internet",
        "iv10r5c1": "¿Usa internet para redes sociales?",
        "iv12r2c1": "Redes para ver contenidos de otros y comentar",
        "iv12r13c1": "Redes para entretenimiento (seguir influenciadores)",
    }
    df_dane_sel = df_dane[list(VARIABLES_DANE)]
    display(df_dane_sel.describe())

,p1r1c1,p2r1c1,iii4r1c1,iii7r1c1,iv5r1c1,iv10r5c1,iv12r2c1,iv12r13c1
count,148700.000000,148700.000000,148700.000000,148700.000000,139223.000000,99064.000000,80840.000000,80840.000000
mean,44.774264,45062.669274,1.522024,35.626873,2.368423,1.183962,1.467900,1.224171
std,29.673332,29660.203245,0.499516,22.313871,1.730378,0.387455,0.498972,0.417038
min,5.000000,5001.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000
25%,17.000000,17867.000000,1.000000,16.000000,1.000000,1.000000,1.000000,1.000000
50%,44.000000,44430.000000,2.000000,34.000000,1.000000,1.000000,1.000000,1.000000
75%,70.000000,70670.000000,2.000000,53.000000,5.000000,1.000000,2.000000,1.000000
max,99.000000,99773.000000,2.000000,107.000000,5.000000,2.000000,2.000000,2.000000


**Interpretación.**
- `iii7r1c1` (edad): va de **0 a 107 años**. El DANE encuesta a todas las personas del hogar, no solo a jóvenes; para el proyecto habría que filtrar 18 a 25.
- `iii4r1c1` va de 1 a 2 (Hombre, Mujer), coherente con el diccionario.
- `iv10r5c1`, `iv12r2c1` e `iv12r13c1` van de 1 a 2 (Sí, No) y tienen `count` menores que 148.700: confirman preguntas condicionadas.
- `p2r1c1` (municipio) tiene como mínimo `5001`: es el código de Medellín (05001) **sin el cero inicial**, porque se guardó como número.

In [35]:
if df_dane is not None:
    columnas_texto = df_dane.select_dtypes(include="object").columns
    display(df_dane[columnas_texto].describe(include="object"))

C:\Users\maile\AppData\Local\Temp\ipykernel_5308\2717524219.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columnas_texto = df_dane.select_dtypes(include="object").columns
C:\Users\maile\AppData\Local\Temp\ipykernel_5308\2717524219.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_

,UID_ENCUESTA,uid_padre,UUID,UUID_1,FEX_c
count,148700,148700,148700,148700,148700
unique,53343,53343,53570,148700,38842
top,9b1b1ce8-a8cc-4e98-bd52-1631f9db5270,9b1b1ce8-a8cc-4e98-bd52-1631f9db5270,9b1b1ce8-a8cc-4e98-bd52-1631f9db5270/Hogares[1],a0dc1da3-ce83-4d87-860e-d451da72dbc4/Hogares[1]/Personas[1],"32,93505021"
freq,16,16,16,1,151


**Interpretación.**
- `UUID_1` tiene **148.700 valores únicos**, uno por fila: es el identificador de cada persona (termina en `/Personas[n]`).
- `UID_ENCUESTA` tiene 53.343 valores únicos y `UUID` (que termina en `/Hogares[n]`) tiene 53.570. Esto confirma que varias personas comparten encuesta y hogar.
- `FEX_c` tiene 38.842 valores distintos y el más frecuente (`32,93505021`) se repite 151 veces. Todos sus valores son texto con coma decimal.

El aviso `Pandas4Warning` es el mismo que se explicó en Pritchard.

## 4. Tabla de diagnóstico estructural

La tabla se construye con los resultados de `shape` y `dtypes` de cada DataFrame, para que las cifras salgan directamente de los datos.

In [36]:
def resumen_estructural(nombre, df):
    tipos = ", ".join(f"{tipo}: {n}" for tipo, n in df.dtypes.astype(str).value_counts().items())
    return {"Dataset": nombre, "Filas": df.shape[0], "Columnas": df.shape[1], "Tipos de datos": tipos}


resumenes = [resumen_estructural("Pritchard (2021)", df_pritchard),
             resumen_estructural("Engeln et al. (2020)", df_engeln)]
if df_dane is not None:
    resumenes.append(resumen_estructural("DANE ENTIC Hogares 2024", df_dane))

tabla_estructural = pd.DataFrame(resumenes)
(RAIZ / "reports").mkdir(exist_ok=True)
tabla_estructural.to_csv(RAIZ / "reports" / "tabla_estructural.csv", index=False)
tabla_estructural

,Dataset,Filas,Columnas,Tipos de datos
0,Pritchard (2021),404,246,"float64: 221, str: 25"
1,Engeln et al. (2020),308,70,float64: 70
2,DANE ENTIC Hogares 2024,148700,377,"float64: 347, int64: 25, str: 5"


| Dataset | Observaciones |
|---|---|
| Pritchard (2021) | Cada fila es una persona encuestada. Las respuestas cerradas son **códigos guardados como `float64`**; necesitan el diccionario para interpretarse. Las respuestas abiertas vacías están como texto `""` y `info()` no las cuenta como faltantes. Hay edades de 17 y 26 años y bloques de preguntas con muchas menos respuestas (Twitter, Snapchat). |
| Engeln et al. (2020) | Cada fila es una participante del experimento, ordenada por grupo (`Condition`). Todo es `float64`; no tiene texto. Las variables de la tarea tienen unos 106 faltantes, probablemente del grupo de control. Hay al menos una edad de 26 años. |
| DANE ENTIC Hogares 2024 | Cada fila es una persona; las filas no son hogares (53.343 encuestas). Nombres de columna en código y en minúscula, distintos del diccionario. `FEX_c` es texto con coma decimal. Muchas columnas con faltantes por preguntas condicionadas y edades de 0 a 107 años. Solo aporta contexto de exposición. |

En la sección 5 se cuantifican estos puntos y se proponen las acciones de limpieza.

## 5. Diagnóstico inicial de calidad

Esta sección responde al Momento 4 de la EV06. Para cada dataset se revisan:

1. **Valores faltantes**, separando los **justificados** (preguntas que la persona no debía responder) de los **no esperados**.
2. **Tipos de datos:** si el tipo coincide con lo que representa la variable.
3. **Inconsistencias:** códigos que no existen en el diccionario, etiquetas desordenadas, edades fuera de rango, identificadores repetidos.
4. **Limitaciones** para responder la pregunta analítica.
5. **Acciones de limpieza propuestas.** Solo se proponen: en esta actividad **no se modifica ningún dato**.

Toda la evidencia sale de las celdas de esta sección.

In [37]:
def rango_edad(edad):
    """Versión de 00_entorno.ipynb que además separa los valores vacíos."""
    if pd.isna(edad):
        return "sin dato"
    if edad < 18 or edad > 25:
        return "fuera de rango"
    elif edad <= 20:
        return "18-20"
    elif edad <= 23:
        return "21-23"
    else:
        return "24-25"


def pregunta(meta, variable, largo=70):
    """Texto de la pregunta (los últimos caracteres si es muy largo)."""
    texto = meta.column_names_to_labels.get(variable) or ""
    return texto if len(texto) <= largo else "…" + texto[-largo:]


def resumen_nulos(df, meta=None):
    """Columnas con nulos, ordenadas de más a menos."""
    nulos = df.isna().sum()
    tabla = pd.DataFrame({"nulos": nulos, "porcentaje": (nulos / len(df) * 100).round(1)})
    if meta is not None:
        tabla.insert(0, "pregunta", [pregunta(meta, c) for c in tabla.index])
    return tabla[tabla["nulos"] > 0].sort_values("nulos", ascending=False)


def respuestas_por_filtro(df, filtro, pregunta_condicionada):
    """Cuántas filas respondieron una pregunta según la respuesta a la pregunta filtro (-1 = filtro vacío)."""
    return df.groupby(df[filtro].fillna(-1))[pregunta_condicionada].agg(filas="size", respondieron="count")

`rango_edad` es la función de `00_entorno.ipynb` con un cambio: si la edad está vacía devuelve `"sin dato"`. Sin ese cambio, una edad vacía (`NaN`) no cumple ninguna comparación y terminaría clasificada en `"24-25"`, que es la rama `else`.

### 5.1 Valores faltantes

#### Pritchard

In [38]:
nulos_pritchard = resumen_nulos(df_pritchard, meta_pritchard)
print("Columnas con al menos un nulo:", len(nulos_pritchard), "de", df_pritchard.shape[1])
print("Celdas nulas en total:", int(nulos_pritchard["nulos"].sum()))
nulos_pritchard.head(10)

Columnas con al menos un nulo: 220 de 246
Celdas nulas en total: 3130


,pregunta,nulos,porcentaje
Q41,Do you edit your photos or use filters?,350,86.6
Q44,Does it matter if people comment on the image of yourself?,350,86.6
Q35,How many times a day do you go on Twitter?,225,55.7
Q34,What is the average amount of time you spend on Twitter ...,220,54.5
Q38,How often do you post images of yourself on Twitter?,220,54.5
Q33,Is there a specific time of day you generally post on Tw...,220,54.5
Q57,Do you edit your photos or use filters?,117,29.0
Q60,Does it matter if people comment on the image of yourself?,116,28.7
Q51,How many times a day do you go on Snapchat?,55,13.6
Q50,What is the average amount of time you spend on Snapchat...,50,12.4


**Interpretación.** Las columnas con más nulos pertenecen a los bloques de Twitter (`Q33`–`Q44`). Antes de llamarlos "datos perdidos" hay que comprobar si esas preguntas eran **condicionadas**, es decir, si solo se le hacían a quien usa Twitter.

In [39]:
print("Q32:", pregunta(meta_pritchard, "Q32"))
display(etiquetas(dic_pritchard, "Q32"))
print("Q34 según la respuesta a Q32:")
display(respuestas_por_filtro(df_pritchard, "Q32", "Q34"))
print("Q41 según la frecuencia con que publica fotos propias en Twitter (Q38):")
display(etiquetas(dic_pritchard, "Q38"))
display(respuestas_por_filtro(df_pritchard, "Q38", "Q41"))
print("Q17 (tiempo en Instagram) según Q16 (¿usa Instagram?):")
display(respuestas_por_filtro(df_pritchard, "Q16", "Q17"))

Q32: Do you use Twitter?


,codigo,etiqueta_valor
478,1.0,Yes
479,2.0,No


Q34 según la respuesta a Q32:


,filas,respondieron
Q32,,
-1.0,1,0
1.0,184,184
2.0,219,0


Q41 según la frecuencia con que publica fotos propias en Twitter (Q38):


,codigo,etiqueta_valor
502,1.0,Never
503,2.0,Rarely
504,3.0,Sometimes
505,4.0,Frequently
506,5.0,Every day


,filas,respondieron
Q38,,
-1.0,220,0
1.0,130,0
2.0,50,50
3.0,4,4


Q17 (tiempo en Instagram) según Q16 (¿usa Instagram?):


,filas,respondieron
Q16,,
1.0,385,385
2.0,19,0


**Interpretación.** Los faltantes de estos bloques son **justificados**:
- `Q34` (tiempo en Twitter) solo tiene respuesta cuando `Q32 = 1` (*Yes*). Quienes no usan Twitter no la respondieron.
- `Q41` (¿editas tus fotos?) solo la respondieron usuarios que publican fotos propias en Twitter (`Q38` = 2 o 3). Por eso tiene solo 54 respuestas.
- `Q17` tiene respuesta para todas las personas que usan Instagram y para ninguna que no lo usa.

**Impacto:** si se cuentan como olvidos, se exagera la falta de respuesta. Los porcentajes de estas preguntas deben calcularse **solo sobre quienes debían responderlas**.

In [40]:
columnas_texto_p = df_pritchard.select_dtypes(include=["str"]).columns
vacios_pritchard = df_pritchard[columnas_texto_p].apply(lambda s: s.str.strip().eq("")).sum()
celdas_texto_p = len(columnas_texto_p) * len(df_pritchard)
mitad_vacias_p = int((vacios_pritchard > len(df_pritchard) / 2).sum())

print("Columnas de texto:", len(columnas_texto_p))
print("Celdas de texto vacías:", int(vacios_pritchard.sum()), "de", celdas_texto_p)
print("Columnas de texto con más de la mitad vacía:", mitad_vacias_p)
print("Nulos que info() reporta en esas columnas:", int(df_pritchard[columnas_texto_p].isna().sum().sum()))
vacios_pritchard.sort_values(ascending=False).head(8).to_frame("celdas vacías")

Columnas de texto:

 25
Celdas de texto vacías: 6041 de 10100
Columnas de texto con más de la mitad vacía: 16


Nulos que info() reporta en esas columnas: 0


,celdas vacías
Q36,398
Q45,397
Q46,397
Q53,389
Q43,380
Q42,380
Q62,367
Q61,367


**Interpretación.** Este es un faltante **oculto**. Las respuestas abiertas sin contestar están guardadas como texto vacío (`""`), no como `NaN`. Por eso `info()` reporta **0 nulos** en esas columnas, aunque más de la mitad de las celdas de texto estén vacías.

**Impacto:** cualquier conteo de faltantes con `isna()` daría un resultado falso. Además, `describe(include="object")` toma el texto vacío como la respuesta más frecuente.

#### Engeln

In [41]:
VARIABLES_TAREA = ["imagepeople", "imageobject", "reading", "videopeople", "videoobject", "other",
                   "overallappcompduringtask", "specbodypartcompduringtask", "comparisonsduringtask"]

display(resumen_nulos(df_engeln, meta_engeln).head(12))
print("Nulos de las variables de la tarea por grupo (Condition):")
df_engeln.groupby("Condition")[VARIABLES_TAREA].apply(lambda d: d.isna().sum())

,pregunta,nulos,porcentaje
imageobject,percent time looking at images not featuring people,107,34.7
other,percent time doing something else,107,34.7
videoobject,percent time looking at videos of objects,107,34.7
comparisonsduringtask,total score for comparisons during task,106,34.4
specbodypartcompduringtask,…u compare specific body parts to people you saw on Inst...,106,34.4
overallappcompduringtask,…mpare your overall appearance to people you saw on Inst...,106,34.4
imagepeople,percent time looking at images of people,106,34.4
reading,percent time reading or commenting,106,34.4
videopeople,percent time looking at videos of people,106,34.4
BISSTotalPre,biss pre-test,8,2.6


Nulos de las variables de la tarea por grupo (Condition):


,imagepeople,imageobject,reading,videopeople,videoobject,other,overallappcompduringtask,specbodypartcompduringtask,comparisonsduringtask
Condition,,,,,,,,,
1.0,0,1,0,0,0,0,0,0,0
2.0,0,0,0,0,1,1,0,0,0
3.0,106,106,106,106,106,106,106,106,106


**Interpretación.** Las variables de la tarea (qué miró y cuánto se comparó en la red social) tienen **106 nulos, y todos están en `Condition = 3`**, el grupo que jugó Bejeweled. Ese grupo no usó redes, así que estos faltantes son **justificados por el diseño del experimento**. Fuera del grupo de control solo hay 3 nulos (`imageobject`, `videoobject`, `other`), que sí son faltantes no esperados.

In [42]:
items_biss_pre = [f"BISS{i}pre" for i in range(1, 7)]
faltan_items_pre = df_engeln[items_biss_pre].isna().sum(axis=1)

print("Filas con algún ítem BISS 'pre' vacío:", int((faltan_items_pre > 0).sum()))
print("Filas con BISSTotalPre vacío:", int(df_engeln["BISSTotalPre"].isna().sum()))
print("¿Son las mismas filas?", bool(((faltan_items_pre > 0) == df_engeln["BISSTotalPre"].isna()).all()))

Filas con algún ítem BISS 'pre' vacío: 8
Filas con BISSTotalPre vacío: 8
¿Son las mismas filas? True


**Interpretación.** El puntaje total de satisfacción inicial (`BISSTotalPre`) está vacío exactamente en las filas a las que les falta al menos un ítem. Es un faltante **no esperado pero explicable**: se calculó el total solo con los ítems completos.

#### DANE

In [43]:
if df_dane is not None:
    nulos_dane = df_dane.isna().sum()
    print("Columnas completas:", int((nulos_dane == 0).sum()))
    print("Columnas con nulos:", int((nulos_dane > 0).sum()))
    print("Columnas con más de la mitad vacía:", int((nulos_dane > len(df_dane) / 2).sum()))
    print("Columnas totalmente vacías:", nulos_dane[nulos_dane == len(df_dane)].index.tolist())

Columnas completas: 30
Columnas con nulos: 347
Columnas con más de la mitad vacía: 194
Columnas totalmente vacías: ['iv13r20c2']


In [44]:
if df_dane is not None:
    print("IV5R1C1:", dic_dane.loc[dic_dane["variable"] == "IV5R1C1", "etiqueta_variable"].iloc[0])
    display(etiquetas(dic_dane, "IV5R1C1"))
    print("iv10r5c1 (¿usa internet para redes sociales?) según iv5r1c1:")
    display(respuestas_por_filtro(df_dane, "iv5r1c1", "iv10r5c1"))
    coincide_salto = (df_dane["iv12r13c1"].notna() == (df_dane["iv10r5c1"] == 1)).all()
    print("iv12r13c1 tiene respuesta si y solo si iv10r5c1 = 1 (Sí):", bool(coincide_salto))

IV5R1C1: ¿Con qué frecuencia utiliza _____  Internet (en cualquier lugar y desde cualquier dispositivo)?


,codigo,etiqueta_valor
306,1,Todos los días de la semana
307,2,"Al menos una vez a la semana, pero no cada día"
308,3,"Al menos una vez al mes, pero no cada semana"
309,4,"Al menos una vez al año, pero no cada mes"
310,5,No utiliza internet


iv10r5c1 (¿usa internet para redes sociales?) según iv5r1c1:


,filas,respondieron
iv5r1c1,,
-1.0,9477,0
1.0,71564,71564
2.0,25311,25311
3.0,1998,1998
4.0,191,191
5.0,40159,0


iv12r13c1 tiene respuesta si y solo si iv10r5c1 = 1 (Sí): True


**Interpretación.** En el DANE la mayoría de los faltantes son **saltos del cuestionario**:
- `iv10r5c1` solo se pregunta a quien usa internet. No tienen respuesta las filas con `iv5r1c1 = 5` (*No utiliza internet*) ni las filas con `iv5r1c1` vacío; las que tienen los códigos 1 a 4 responden todas.
- `iv12r13c1` (redes para entretenimiento) tiene respuesta **en todas y solo en** las filas donde `iv10r5c1 = 1` (usa redes sociales).
- La columna `iv13r20c2` está completamente vacía y no aporta información.

### 5.2 Tipos de datos

In [45]:
def resumen_tipos_codigos(df, meta):
    numericas = df.select_dtypes("number").columns
    categoricas = [c for c in numericas if c in meta.variable_value_labels]
    enteros = all((df[c].dropna() % 1 == 0).all() for c in categoricas)
    return {"columnas numéricas (float64)": len(numericas),
            "con etiquetas de valor (son categorías)": len(categoricas),
            "sin etiquetas de valor": len(numericas) - len(categoricas),
            "todos los códigos son enteros": enteros}


tipos_codigos = pd.DataFrame({"Pritchard": resumen_tipos_codigos(df_pritchard, meta_pritchard),
                              "Engeln": resumen_tipos_codigos(df_engeln, meta_engeln)})
tipos_codigos

,Pritchard,Engeln
columnas numéricas (float64),221,70
con etiquetas de valor (son categorías),200,56
sin etiquetas de valor,21,14
todos los códigos son enteros,True,True


**Interpretación.** En los dos `.sav`, la mayoría de las columnas numéricas tienen etiquetas de valor: son **categorías codificadas** (Sí/No, Nunca…Siempre), aunque pandas las guarde como `float64`. Todos los códigos son números enteros; aparecen con decimal (`2.0`) porque SPSS guarda todo como número decimal y `float64` admite `NaN`.

**Impacto:** con este tipo, pandas permite calcular promedios de categorías sin advertir nada.

In [46]:
if df_dane is not None:
    fex_numerico = pd.to_numeric(df_dane["FEX_c"].str.replace(",", ".", regex=False), errors="coerce")
    print("Tipo actual de FEX_c:", df_dane["FEX_c"].dtype)
    print("Valores convertibles a número cambiando la coma por punto:", int(fex_numerico.notna().sum()), "de", len(df_dane))
    print(f"Suma del factor de expansión (personas que representa la muestra): {fex_numerico.sum():,.0f}")

    tipos_declarados = dic_dane.drop_duplicates("variable")["tipo_dato"].value_counts()
    print("Tipos declarados en el diccionario oficial:", tipos_declarados.to_dict())
    print("Filas con código de departamento de un dígito (p1r1c1 < 10):", int((df_dane["p1r1c1"] < 10).sum()))
    display(df_dane.loc[df_dane["p2r1c1"] == 5001, ["p1r1c1", "p2r1c1"]].head(1))

Tipo actual de FEX_c: str
Valores convertibles a número cambiando la coma por punto: 148700 de 148700
Suma del factor de expansión (personas que representa la muestra): 52,933,447
Tipos declarados en el diccionario oficial: {'VARCHAR2': 376, 'NUMBER': 1}
Filas con código de departamento de un dígito (p1r1c1 < 10): 14494


,p1r1c1,p2r1c1
10996,5,5001


**Interpretación.**
- `FEX_c` es texto porque usa **coma decimal**. Cambiando la coma por punto, el 100 % de los valores se convierte en número, y la suma da unos 52,9 millones de personas, una magnitud coherente con la población de Colombia. La conversión es segura, pero **mientras sea texto no se puede ponderar**.
- El diccionario del DANE declara casi todas las variables como texto (`VARCHAR2`), pero pandas las leyó como números. Así se perdieron los **ceros iniciales** de los códigos territoriales: Antioquia (`05`) aparece como `5` y Medellín (`05001`) como `5001`.

### 5.3 Posibles inconsistencias

#### Pritchard: códigos que no existen en el diccionario

In [47]:
def codigos_fuera_de_etiquetas(df, meta):
    """Variables con valores observados que no tienen etiqueta en el diccionario SPSS."""
    filas = []
    for variable, etiquetas_valor in meta.variable_value_labels.items():
        valores = df[variable].dropna()
        fuera = valores[~valores.isin(list(etiquetas_valor))]
        if len(fuera):
            filas.append({"variable": variable,
                          "codigos_en_etiquetas": f"{min(etiquetas_valor):g}–{max(etiquetas_valor):g}",
                          "valores_observados": f"{valores.min():g}–{valores.max():g}",
                          "celdas_sin_etiqueta": len(fuera)})
    return pd.DataFrame(filas)


fuera_pritchard = codigos_fuera_de_etiquetas(df_pritchard, meta_pritchard)
print("Variables con valores sin etiqueta:", len(fuera_pritchard))
print("Celdas afectadas:", int(fuera_pritchard["celdas_sin_etiqueta"].sum()))
(fuera_pritchard.assign(bloque=fuera_pritchard["variable"].str.split("_").str[0])
 .groupby("bloque")
 .agg(variables=("variable", "size"),
      codigos_en_etiquetas=("codigos_en_etiquetas", lambda s: ", ".join(sorted(set(s)))),
      valores_observados=("valores_observados", lambda s: ", ".join(sorted(set(s)))),
      celdas=("celdas_sin_etiqueta", "sum")))

Variables con valores sin etiqueta: 97
Celdas afectadas: 14503


,variables,codigos_en_etiquetas,valores_observados,celdas
bloque,,,,
Q15,26,1–6,0–3,8234
Q23,7,1–6,0–3,1732
Q64,24,2–6,1–5,972
Q65,40,1–5,0–4,3565


In [48]:
items_q65 = [c for c in df_pritchard.columns if c.startswith("Q65_")]
print("Ejemplo de ítem Q65:", pregunta(meta_pritchard, "Q65_1", 90))
display(etiquetas(dic_pritchard, "Q65_1"))
valores_q65 = sorted(int(v) for v in pd.unique(df_pritchard[items_q65].to_numpy().ravel()) if pd.notna(v))
print("Valores observados en los", len(items_q65), "ítems de Q65:", valores_q65)
print("Rango de PACSR (puntaje de comparación):", df_pritchard["PACSR"].min(), "a", df_pritchard["PACSR"].max())

Ejemplo de ítem Q65: … of comparisons. - When I’m at the gym, I compare my body size to the body size of others.


,codigo,etiqueta_valor
853,1.0,Never
854,2.0,Seldom
855,3.0,Sometimes
856,4.0,Often
857,5.0,Always


Valores observados en los 40 ítems de Q65: [0, 1, 2, 3, 4]
Rango de PACSR (puntaje de comparación): 0.0 a 4.0


**Interpretación.** Esta es la inconsistencia **más importante** para el proyecto, porque afecta a la dimensión de **comparación**. En los ítems de `Q65`, el diccionario va de `1 = Never` a `5 = Always`, pero los datos van de **0 a 4**: el código 0 no tiene etiqueta y el código 5 nunca aparece. El puntaje `PACSR` también va de 0 a 4. Lo más probable es que los datos se hayan recodificado a 0–4 sin actualizar las etiquetas, es decir, que el diccionario esté **corrido una posición**. `Q64` muestra lo mismo (datos 1–5, etiquetas 2–6). En los bloques `Q15` y `Q23` (alimentación) también hay valores sin etiqueta.

**Impacto:** si se interpreta con el diccionario actual, `Q65 = 1` se leería como *Never* cuando probablemente significa *Seldom*. Todas las frecuencias de comparación quedarían mal descritas.

#### Pritchard: escalas incompletas o desordenadas

In [49]:
print("Q17:", pregunta(meta_pritchard, "Q17"))
display(etiquetas(dic_pritchard, "Q17").head(4))
print("Q17_1:", pregunta(meta_pritchard, "Q17_1", 90))
display(etiquetas(dic_pritchard, "Q17_1"))

Q17: What is the average amount of time you spend on Instagram each day?


,codigo,etiqueta_valor
14,1.0,0-15 min
15,2.0,15-30 min
16,3.0,1-2 hr
17,4.0,2-3 hr


Q17_1: … months have you: - Gone on eating binges where you feel that you may not be able to stop?


,codigo,etiqueta_valor
450,1.0,Never
451,2.0,Once a month or less
452,3.0,2-3 times a month
453,4.0,Once a month
454,5.0,2-6 times a week
455,6.0,Once a day or more


**Interpretación.**
- En `Q17` (tiempo en Instagram) se pasa de `2 = 15-30 min` a `3 = 1-2 hr`: **no existe la opción de 30 a 60 minutos**. Quien usaba Instagram unos 45 minutos tuvo que elegir una opción que no le correspondía.
- En `Q17_1` el código 4 dice *Once a month*, después de *2-3 times a month* y antes de *2-6 times a week*. El orden lógico sugiere que debería decir *Once a week*.

#### Pritchard: edades, identificadores y variables redundantes

In [50]:
print("Edades según los rangos del EV02:")
display(df_pritchard["age"].map(rango_edad).value_counts())
print("Edades fuera de rango:", df_pritchard.loc[df_pritchard["age"].map(rango_edad) == "fuera de rango", "age"].value_counts().to_dict())

Edades según los rangos del EV02:


age
18-20             359
21-23              26
fuera de rango     11
24-25               7
sin dato            1
Name: count, dtype: int64

Edades fuera de rango: {17.0: 9, 26.0: 2}


In [51]:
filas_id_repetido = df_pritchard[df_pritchard["id"].duplicated(keep=False)]
print("Valores de id repetidos:", filas_id_repetido["id"].unique().tolist(), "→", len(filas_id_repetido), "filas")
print("Filas duplicadas completas (duplicated):", int(df_pritchard.duplicated().sum()))
columnas_distintas_id = int((filas_id_repetido.drop(columns="id").nunique(dropna=False) > 1).sum())
print("Columnas en las que esas filas tienen valores distintos:", columnas_distintas_id)

Valores de id repetidos: ['10720'] → 2 filas


Filas duplicadas completas (duplicated): 0
Columnas en las que esas filas tienen valores distintos: 204


**Interpretación.** No hay filas duplicadas, pero el `id` **10720 aparece en dos filas** con respuestas distintas en muchas columnas. No es una respuesta enviada dos veces: son dos personas con el mismo identificador, probablemente por un error al asignarlo.

In [52]:
print("GenderRE:"); display(etiquetas(dic_pritchard, "GenderRE"))
print("filter_$:", pregunta(meta_pritchard, "filter_$")); display(etiquetas(dic_pritchard, "filter_$"))
display(pd.crosstab(df_pritchard["gender"].fillna(-1), df_pritchard["GenderRE"].fillna(-1)))
print("filter_$ = 1:", int((df_pritchard["filter_$"] == 1).sum()), "| gender = 2 (Female):", int((df_pritchard["gender"] == 2).sum()))

GenderRE:


,codigo,etiqueta_valor
1076,0.0,female
1077,1.0,male


filter_$: gender=2 (FILTER)


,codigo,etiqueta_valor
1065,0.0,Not Selected
1066,1.0,Selected


GenderRE,-1.0,0.0,1.0
gender,,,
-1.0,1,0,0
1.0,0,0,163
2.0,0,235,0
3.0,2,0,0
4.0,3,0,0


filter_$ = 1: 235 | gender = 2 (Female): 235


**Interpretación.** El género está guardado **tres veces**: `gender` (1 = Male, 2 = Female, 3 = Transgender, 4 = Other), `GenderRE` (0 = female, 1 = male) y `filter_$`, una variable técnica que SPSS crea al filtrar (`gender=2 (FILTER)`). `GenderRE` es coherente con `gender`, pero deja **vacías** a las 5 personas que respondieron *Transgender* u *Other*. `filter_$` marca a las mismas 235 personas que `gender = 2`, así que no aporta información nueva.

#### Engeln: etiquetas de la escala BISS

In [53]:
for variable in ["BISS4pre", "BISS5pre"]:
    print(variable)
    display(etiquetas(dic_engeln, variable))

BISS4pre


,codigo,etiqueta_valor
130,1.0,Extremely physically attractive
131,2.0,Very physically attractive
132,3.0,Moderately physically attractive
133,4.0,Moderately physically unattractive
134,5.0,Slightly physically attractive
135,6.0,Neither attractive nor unattractive
136,7.0,Slightly physically unattractive
137,8.0,Very physically unattractive
138,9.0,Extremely physically unattractive


BISS5pre


,codigo,etiqueta_valor
139,1.0,A great deal worse about my looks than I usually feel
140,2.0,Much worse about my looks than I usually feel
141,3.0,About the same about my looks as usual
142,4.0,Somewhat worse about my looks than I usually feel
143,5.0,Just slightly worse about my looks than I usually feel
144,6.0,Just slightly better about my looks than I usually feel
145,7.0,Somewhat better about my looks than I usually feel
146,8.0,Much better about my looks than I usually feel
147,9.0,A great deal better about my looks than I usually feel


**Interpretación.** Las etiquetas de los ítems BISS (satisfacción con la apariencia, escala de 1 a 9) **no siguen un orden lógico**:
- `BISS4pre`: `4 = Moderately physically unattractive` está entre `3 = Moderately… attractive` y `5 = Slightly… attractive`.
- `BISS5pre`: `3 = About the same` está entre `2 = Much worse` y `4 = Somewhat worse`.

Parece que algunas etiquetas quedaron en una posición equivocada al crear el archivo.

In [54]:
for variable in ["BISS1pre", "BISS2pre"]:
    extremos = etiquetas(dic_engeln, variable).iloc[[0, -1]]
    print(variable, "→", " | ".join(f"{c:g} = {t}" for c, t in zip(extremos["codigo"], extremos["etiqueta_valor"])))

promedio_simple = df_engeln[items_biss_pre].mean(axis=1).round(4)
print("Filas donde BISSTotalPre es el promedio simple de los 6 ítems:",
      int((promedio_simple == df_engeln["BISSTotalPre"].round(4)).sum()), "de", int(df_engeln["BISSTotalPre"].notna().sum()))

BISS1pre → 1 = Extremely dissatisfied with my physical appearance | 9 = Extremely satisfied with my physical appearance
BISS2pre → 1 = Extremely satisfied with my body size and shape | 9 = Extremely dissatisfied with my body size and shape
Filas donde BISSTotalPre es el promedio simple de los 6 ítems: 0 de 300


**Interpretación.** Los ítems van en **direcciones opuestas**: en `BISS1pre` el 1 es *muy insatisfecha* y en `BISS2pre` el 1 es *muy satisfecha*. `BISSTotalPre` no coincide con el promedio simple de los ítems en ninguna fila, lo que indica que algunos ítems se invirtieron antes de calcular el total. El archivo **no documenta** cómo se hizo.

In [55]:
print("appearancethoughts:", pregunta(meta_engeln, "appearancethoughts"))
display(etiquetas(dic_engeln, "appearancethoughts"))
display(df_engeln["appearancethoughts"].value_counts().sort_index())

porcentajes = ["imagepeople", "imageobject", "reading", "videopeople", "videoobject", "other"]
suma_porcentajes = df_engeln.loc[df_engeln["Condition"] < 3, porcentajes].sum(axis=1)
print("Suma de porcentajes de tiempo en los grupos 1 y 2:", suma_porcentajes.describe()[["count", "min", "max"]].to_dict())
print("Edades según los rangos del EV02:")
display(df_engeln["Age"].map(rango_edad).value_counts())

appearancethoughts: appearance thoughts during task


,codigo,etiqueta_valor
323,1.0,no thought about my appearance
324,7.0,a lot of thought about my appearance


appearancethoughts
1.0    128
2.0     53
3.0     47
4.0     31
5.0     34
6.0      8
7.0      7
Name: count, dtype: int64

Suma de porcentajes de tiempo en los grupos 1 y 2: {'count': 202.0, 'min': 100.0, 'max': 100.0}
Edades según los rangos del EV02:


Age
18-20             245
21-23              62
fuera de rango      1
Name: count, dtype: int64

**Interpretación.**
- `appearancethoughts` solo tiene etiquetadas las puntas (1 y 7). Los valores 2 a 6 no tienen texto, pero es normal en una escala anclada en los extremos. No es un error, aunque conviene documentarlo.
- Los seis porcentajes de tiempo suman **exactamente 100** en todas las filas de los grupos 1 y 2. Este es un **control de coherencia que se cumple**.
- Hay **una participante de 26 años**, fuera del rango del proyecto.

#### DANE: dominios, nombres y edades

In [56]:
def fuera_de_dominio_dane(df, dic, variables):
    filas = []
    for variable in variables:
        dominio = {int(c) for c in dic.loc[dic["variable"].str.lower() == variable, "codigo"].dropna()}
        if dominio:
            observados = {int(v) for v in df[variable].dropna().unique()}
            filas.append({"variable": variable, "dominio": sorted(dominio),
                          "observados": sorted(observados), "fuera_de_dominio": sorted(observados - dominio)})
    return pd.DataFrame(filas)


if df_dane is not None:
    display(fuera_de_dominio_dane(df_dane, dic_dane, list(VARIABLES_DANE)))
    no_minuscula = [c for c in df_dane.columns if not c.islower()]
    print("Columnas del CSV que no están en minúscula:", no_minuscula)
    print("Ejemplo en el diccionario:", dic_dane["variable"].drop_duplicates().iloc[5:8].tolist())
    print("Edades según los rangos del EV02:")
    display(df_dane["iii7r1c1"].map(rango_edad).value_counts())

,variable,dominio,observados,fuera_de_dominio
0,iii4r1c1,"[1, 2]","[1, 2]",[]
1,iv5r1c1,"[1, 2, 3, 4, 5]","[1, 2, 3, 4, 5]",[]
2,iv10r5c1,"[1, 2]","[1, 2]",[]
3,iv12r2c1,"[1, 2]","[1, 2]",[]
4,iv12r13c1,"[1, 2]","[1, 2]",[]


Columnas del CSV que no están en minúscula: ['ID', 'UID_ENCUESTA', 'UUID', 'UUID_1', 'FEX_c']
Ejemplo en el diccionario: ['P1R1C1', 'P2R1C1', 'P3R1C1']
Edades según los rangos del EV02:


iii7r1c1
fuera de rango    131652
18-20               6273
21-23               6264
24-25               4511
Name: count, dtype: int64

**Interpretación.**
- Las variables pertinentes **no tienen códigos fuera de su dominio**. Este control se cumple.
- Los nombres del CSV están en minúscula (salvo 5 columnas), y el diccionario los escribe en mayúscula. Para unirlos hay que normalizar los nombres.
- La mayoría de las filas está fuera de 18 a 25 años, porque el DANE encuesta a todas las personas del hogar.

### 5.4 Limitaciones de los datos para responder la pregunta

In [57]:
patron = r"makeup|make-up|cosmetic|fashion|beauty|tiktok|maquillaje|moda|belleza"
for nombre, dic in [("Pritchard", dic_pritchard), ("Engeln", dic_engeln), ("DANE", dic_dane)]:
    if dic is None:
        continue
    texto = dic["etiqueta_variable"].astype(str) + " " + dic["etiqueta_valor"].astype(str)
    variables = dic.loc[texto.str.contains(patron, case=False), "variable"].unique().tolist()
    print(f"{nombre}: variables que mencionan moda, maquillaje, belleza o TikTok → {len(variables)} {variables}")

Pritchard: variables que mencionan moda, maquillaje, belleza o TikTok → 0 []
Engeln: variables que mencionan moda, maquillaje, belleza o TikTok → 0 []
DANE: variables que mencionan moda, maquillaje, belleza o TikTok → 0 []


In [58]:
if df_dane is not None:
    jovenes_dane = df_dane[df_dane["iii7r1c1"].between(18, 25)]
    print("Filas de 18 a 25 años:", len(jovenes_dane))
    print("  … que usan internet para redes sociales (iv10r5c1 = 1):", int((jovenes_dane["iv10r5c1"] == 1).sum()))
    print("  … en Antioquia (p1r1c1 = 5):", int((jovenes_dane["p1r1c1"] == 5).sum()))
    print("  … en Medellín (p2r1c1 = 5001):", int((jovenes_dane["p2r1c1"] == 5001).sum()))

Filas de 18 a 25 años: 17048
  … que usan internet para redes sociales (iv10r5c1 = 1): 14369
  … en Antioquia (p1r1c1 = 5): 1060
  … en Medellín (p2r1c1 = 5001): 186


**Interpretación.** Las cifras del DANE son **filas sin ponderar**: sirven para saber cuántos registros hay disponibles, no para estimar porcentajes de la población (para eso se necesita `FEX_c` convertido a número).

| Limitación | Pritchard | Engeln | DANE |
|---|---|---|---|
| Población | Universitarios de EE. UU. | Mujeres universitarias de EE. UU. | Colombia, todas las edades |
| ¿Mide moda o maquillaje? | No (ver celda anterior) | No | No |
| Exposición | Uso y tiempo en Instagram, Twitter y Snapchat (sin TikTok) | 7 minutos controlados en un experimento | Solo uso general de internet y redes |
| Comparación | Sí (`Q65`, `PACSR`), pero con códigos corridos | Sí, solo durante la tarea | No |
| Resultado percibido | Satisfacción e imagen corporal | Satisfacción antes y después | No |
| Diseño | Encuesta transversal | Experimento | Encuesta transversal oficial |

**Conclusión de las limitaciones:** ningún dataset mide **cambios en preferencias de moda o maquillaje**, y ninguno de los dos de Mendeley representa a jóvenes colombianos. Pritchard y Engeln sirven como **antecedentes con datos reales** para las dimensiones de exposición y comparación; el DANE da contexto de uso de redes en Colombia. La parte central de la pregunta solo podrá responderse con la **encuesta propia**.

### 5.5 Tabla de diagnóstico de calidad

La columna *Evidencia encontrada* se arma con los resultados de las celdas anteriores, para que las cifras coincidan siempre con los datos.

In [59]:
edades_p = df_pritchard["age"].map(rango_edad).value_counts()
edades_e = df_engeln["Age"].map(rango_edad).value_counts()
bloques_fuera = ", ".join(sorted(fuera_pritchard["variable"].str.split("_").str[0].unique()))

filas_calidad = [
    ("Pritchard (2021)", "Respuestas abiertas vacías guardadas como texto, no como nulo",
     f"{int(vacios_pritchard.sum())} de {celdas_texto_p} celdas de texto son \"\"; {mitad_vacias_p} de {len(columnas_texto_p)} columnas de texto tienen más de la mitad vacía; info() reporta 0 nulos en ellas",
     "Se subestiman los faltantes; describe(include='object') toma el vacío como respuesta más frecuente",
     "Reemplazar \"\" y textos solo con espacios por NaN antes de contar faltantes"),
    ("Pritchard (2021)", "Faltantes por preguntas condicionadas (justificados)",
     f"{len(nulos_pritchard)} columnas con nulos; Q34 solo tiene respuesta cuando Q32 = Yes; Q41 solo tiene {int(df_pritchard['Q41'].notna().sum())} respuestas (usuarios que publican fotos en Twitter)",
     "Si se tratan como olvidos, se exagera la falta de respuesta y se calculan porcentajes sobre una base equivocada",
     "Documentar el filtro de cada bloque y calcular porcentajes solo sobre quienes debían responder; no imputar"),
    ("Pritchard (2021)", "Códigos que no coinciden con el diccionario de etiquetas",
     f"{len(fuera_pritchard)} variables y {int(fuera_pritchard['celdas_sin_etiqueta'].sum())} celdas sin etiqueta (bloques {bloques_fuera}); Q65 (comparación) tiene valores 0–4 y etiquetas 1–5; PACSR va de {df_pritchard['PACSR'].min():g} a {df_pritchard['PACSR'].max():g}",
     "Afecta la dimensión de comparación: con el diccionario actual cada respuesta se leería una categoría más abajo",
     "Verificar la codificación con el artículo o instrumento original; corregir las etiquetas (o recodificar) y documentar la decisión"),
    ("Pritchard (2021)", "Respuestas categóricas guardadas como float64",
     f"{tipos_codigos.loc['con etiquetas de valor (son categorías)', 'Pritchard']} de {tipos_codigos.loc['columnas numéricas (float64)', 'Pritchard']} columnas numéricas tienen etiquetas de valor; todos los códigos son enteros",
     "pandas permite promediar categorías; los mean y std de describe() no tienen sentido",
     "Convertir a Int64 o a category (ordenada en las escalas); describir con frecuencias y medianas"),
    ("Pritchard (2021)", "Edades fuera de la población de 18 a 25 años",
     f"{int(edades_p.get('fuera de rango', 0))} fuera de rango (17 y 26 años), {int(edades_p.get('sin dato', 0))} sin dato, {int(edades_p.drop(['fuera de rango', 'sin dato'], errors='ignore').sum())} dentro del rango",
     "Se incluirían personas que no pertenecen a la población del proyecto",
     "Filtrar 18 a 25 años y reportar cuántos registros se excluyen"),
    ("Pritchard (2021)", "Identificador repetido",
     f"id 10720 aparece en {len(filas_id_repetido)} filas con {columnas_distintas_id} columnas distintas; duplicated() = {int(df_pritchard.duplicated().sum())}",
     "El id no identifica a una sola persona; puede causar errores al unir o contar",
     "Crear un identificador propio por fila y registrar el caso; no eliminar ninguna de las dos filas"),
    ("Pritchard (2021)", "Escalas con categorías faltantes o desordenadas",
     "Q17 pasa de '15-30 min' a '1-2 hr' (no hay 30–60 min); en Q17_1 el código 4 'Once a month' está después de '2-3 times a month'",
     "Imprecisión en el tiempo de uso (exposición) y riesgo de ordenar mal una escala",
     "Agrupar Q17 en los rangos del EV02 (<1 h, 1–3 h, >3 h) señalando la limitación; verificar Q17_1 con el instrumento"),
    ("Pritchard (2021)", "Variables redundantes, técnicas o sin documentar",
     f"filter_$ = 1 en {int((df_pritchard['filter_$'] == 1).sum())} filas, igual que gender = 2; GenderRE deja vacías {int((df_pritchard['gender'].isin([3, 4])).sum())} respuestas Transgender/Other; {sum(1 for e in meta_pritchard.column_labels if not e)} variables sin texto de pregunta (puntajes)",
     "Confusión sobre cuál variable usar; pérdida de categorías de género",
     "Usar gender como variable de género; excluir filter_$; documentar los puntajes con el artículo"),
    ("Engeln et al. (2020)", "Faltantes por diseño del experimento (justificados)",
     f"Las variables de la tarea tienen {int((df_engeln['Condition'] == 3).sum())} nulos, todos en Condition = 3 (control); 3 nulos adicionales en los grupos 1 y 2",
     "Si se analizan los tres grupos juntos, el control aparece como 'sin respuesta'",
     "Analizar las variables de la tarea solo para Condition 1 y 2; revisar las 3 filas con faltantes no esperados"),
    ("Engeln et al. (2020)", "Puntajes totales vacíos por ítems faltantes",
     f"BISSTotalPre tiene {int(df_engeln['BISSTotalPre'].isna().sum())} nulos, en las mismas filas con algún ítem BISS 'pre' vacío",
     "Se reduce el número de casos para comparar antes y después",
     "Reportar el número de casos de cada análisis; no imputar"),
    ("Engeln et al. (2020)", "Etiquetas de la escala BISS fuera de orden",
     "BISS4pre: 4 = 'Moderately physically unattractive' entre opciones 'attractive'; BISS5pre: 3 = 'About the same' entre opciones 'worse'",
     "Interpretación incorrecta de la satisfacción con la apariencia",
     "Contrastar con el instrumento BISS original y corregir las etiquetas en el diccionario"),
    ("Engeln et al. (2020)", "Ítems en direcciones opuestas y total sin documentar",
     f"BISS1pre: 1 = muy insatisfecha; BISS2pre: 1 = muy satisfecha; BISSTotalPre coincide con el promedio simple en {int((promedio_simple == df_engeln['BISSTotalPre'].round(4)).sum())} filas",
     "Promediar los ítems sin invertirlos daría puntajes erróneos",
     "Usar BISSTotalPre/Post documentando su cálculo, o invertir los ítems según el instrumento"),
    ("Engeln et al. (2020)", "Edad fuera de rango",
     f"{int(edades_e.get('fuera de rango', 0))} participante de 26 años",
     "Incluye a una persona fuera de la población",
     "Filtrar 18 a 25 años"),
    ("Engeln et al. (2020)", "Escala con etiquetas solo en los extremos",
     f"appearancethoughts solo etiqueta 1 y 7; {int(df_engeln['appearancethoughts'].between(2, 6).sum())} respuestas con valores 2 a 6",
     "Bajo: es una escala anclada, pero el significado intermedio no está escrito",
     "Documentar la escala en el diccionario del proyecto"),
]
if df_dane is not None:
    filas_calidad += [
        ("DANE ENTIC 2024", "Factor de expansión guardado como texto con coma decimal",
         f"FEX_c es {df_dane['FEX_c'].dtype}; {int(fex_numerico.notna().sum())} de {len(df_dane)} valores se convierten cambiando ',' por '.'; suman {fex_numerico.sum():,.0f}",
         "No se puede ponderar; los resultados sin ponderar no representan a la población",
         "Convertir con str.replace(',', '.') y pd.to_numeric antes de cualquier estimación"),
        ("DANE ENTIC 2024", "Códigos categóricos leídos como números (se pierden ceros iniciales)",
         f"El diccionario declara {tipos_declarados.get('VARCHAR2', 0)} variables como VARCHAR2; {int((df_dane['p1r1c1'] < 10).sum())} filas con departamento de un dígito (05 → 5, 05001 → 5001)",
         "Errores al unir con otras fuentes por código DIVIPOLA y riesgo de tratar códigos como cantidades",
         "Leer los códigos territoriales como texto (dtype=str) o completar con zfill; tratar respuestas como category"),
        ("DANE ENTIC 2024", "Nombres de columnas distintos del diccionario",
         f"{sum(c.islower() for c in df_dane.columns)} columnas en minúscula y el diccionario en mayúscula; {len(no_minuscula)} columnas con otro formato ({', '.join(no_minuscula)})",
         "Las búsquedas y uniones con el diccionario fallan si no se normalizan",
         "Normalizar los nombres (por ejemplo, a minúscula) y conservar el nombre original en el diccionario"),
        ("DANE ENTIC 2024", "Faltantes por saltos del cuestionario y una columna vacía",
         f"{int((nulos_dane > 0).sum())} columnas con nulos, {int((nulos_dane > len(df_dane) / 2).sum())} con más de la mitad vacía; iv12r13c1 responde si y solo si iv10r5c1 = 1; iv13r20c2 100 % vacía",
         "Porcentajes mal calculados si se usa el total de filas como base",
         "Calcular cada indicador sobre la población a la que aplica la pregunta; excluir iv13r20c2"),
        ("DANE ENTIC 2024", "Población y cobertura distintas a las del proyecto",
         f"Edades de {int(df_dane['iii7r1c1'].min())} a {int(df_dane['iii7r1c1'].max())} años; {len(jovenes_dane)} filas de 18 a 25 años, {int((jovenes_dane['p2r1c1'] == 5001).sum())} en Medellín; ninguna variable sobre belleza, moda o maquillaje",
         "Solo sirve como contexto de exposición; muestra pequeña para Medellín",
         "Filtrar 18 a 25 años, ponderar con FEX_c y usarlo solo como contexto nacional"),
    ]
filas_calidad.append(
    ("Encuesta propia", "Datos todavía no recolectados",
     "No existe archivo de respuestas en data/raw/encuesta/",
     "No se puede diagnosticar ni responder la parte de moda, maquillaje y población colombiana",
     "Aplicar la encuesta según data/diccionario_datos.md y repetir este diagnóstico con sus respuestas"))

tabla_calidad = pd.DataFrame(filas_calidad, columns=["Dataset", "Problema identificado", "Evidencia encontrada",
                                                     "Posible impacto", "Acción propuesta"])
tabla_calidad.to_csv(RAIZ / "reports" / "tabla_calidad.csv", index=False)
with pd.option_context("display.max_colwidth", None):
    display(tabla_calidad)

,Dataset,Problema identificado,Evidencia encontrada,Posible impacto,Acción propuesta
0,Pritchard (2021),"Respuestas abiertas vacías guardadas como texto, no como nulo","6041 de 10100 celdas de texto son """"; 16 de 25 columnas de texto tienen más de la mitad vacía; info() reporta 0 nulos en ellas",Se subestiman los faltantes; describe(include='object') toma el vacío como respuesta más frecuente,"Reemplazar """" y textos solo con espacios por NaN antes de contar faltantes"
1,Pritchard (2021),Faltantes por preguntas condicionadas (justificados),220 columnas con nulos; Q34 solo tiene respuesta cuando Q32 = Yes; Q41 solo tiene 54 respuestas (usuarios que publican fotos en Twitter),"Si se tratan como olvidos, se exagera la falta de respuesta y se calculan porcentajes sobre una base equivocada",Documentar el filtro de cada bloque y calcular porcentajes solo sobre quienes debían responder; no imputar
2,Pritchard (2021),Códigos que no coinciden con el diccionario de etiquetas,"97 variables y 14503 celdas sin etiqueta (bloques Q15, Q23, Q64, Q65); Q65 (comparación) tiene valores 0–4 y etiquetas 1–5; PACSR va de 0 a 4",Afecta la dimensión de comparación: con el diccionario actual cada respuesta se leería una categoría más abajo,Verificar la codificación con el artículo o instrumento original; corregir las etiquetas (o recodificar) y documentar la decisión
3,Pritchard (2021),Respuestas categóricas guardadas como float64,200 de 221 columnas numéricas tienen etiquetas de valor; todos los códigos son enteros,pandas permite promediar categorías; los mean y std de describe() no tienen sentido,Convertir a Int64 o a category (ordenada en las escalas); describir con frecuencias y medianas
4,Pritchard (2021),Edades fuera de la población de 18 a 25 años,"11 fuera de rango (17 y 26 años), 1 sin dato, 392 dentro del rango",Se incluirían personas que no pertenecen a la población del proyecto,Filtrar 18 a 25 años y reportar cuántos registros se excluyen
5,Pritchard (2021),Identificador repetido,id 10720 aparece en 2 filas con 204 columnas distintas; duplicated() = 0,El id no identifica a una sola persona; puede causar errores al unir o contar,Crear un identificador propio por fila y registrar el caso; no eliminar ninguna de las dos filas
6,Pritchard (2021),Escalas con categorías faltantes o desordenadas,Q17 pasa de '15-30 min' a '1-2 hr' (no hay 30–60 min); en Q17_1 el código 4 'Once a month' está después de '2-3 times a month',Imprecisión en el tiempo de uso (exposición) y riesgo de ordenar mal una escala,"Agrupar Q17 en los rangos del EV02 (<1 h, 1–3 h, >3 h) señalando la limitación; verificar Q17_1 con el instrumento"
7,Pritchard (2021),"Variables redundantes, técnicas o sin documentar","filter_$ = 1 en 235 filas, igual que gender = 2; GenderRE deja vacías 5 respuestas Transgender/Other; 21 variables sin texto de pregunta (puntajes)",Confusión sobre cuál variable usar; pérdida de categorías de género,Usar gender como variable de género; excluir filter_$; documentar los puntajes con el artículo
8,Engeln et al. (2020),Faltantes por diseño del experimento (justificados),"Las variables de la tarea tienen 106 nulos, todos en Condition = 3 (control); 3 nulos adicionales en los grupos 1 y 2","Si se analizan los tres grupos juntos, el control aparece como 'sin respuesta'",Analizar las variables de la tarea solo para Condition 1 y 2; revisar las 3 filas con faltantes no esperados
9,Engeln et al. (2020),Puntajes totales vacíos por ítems faltantes,"BISSTotalPre tiene 8 nulos, en las mismas filas con algún ítem BISS 'pre' vacío",Se reduce el número de casos para comparar antes y después,Reportar el número de casos de cada análisis; no imputar


### 5.6 Acciones de limpieza y preparación propuestas (por prioridad)

Ninguna se ejecuta en esta actividad; quedan propuestas para la etapa de preparación.

| Prioridad | Acción | Dataset | Por qué primero |
|---|---|---|---|
| 1 | Verificar y corregir la correspondencia código ↔ etiqueta de `Q65`, `Q64` (y `Q15`, `Q23`) | Pritchard | Sin esto, la dimensión de comparación se interpreta mal |
| 1 | Corregir las etiquetas BISS y documentar cómo se calculan los totales | Engeln | Afecta la medida de satisfacción con la apariencia |
| 2 | Convertir `""` en `NaN` en las respuestas abiertas | Pritchard | Hace visibles los faltantes reales |
| 2 | Convertir `FEX_c` a número y los códigos territoriales a texto | DANE | Necesario para ponderar y filtrar por territorio |
| 3 | Filtrar 18 a 25 años y reportar cuántos registros se excluyen | Los tres | Delimita la población del proyecto |
| 3 | Definir la base de cada pregunta condicionada (solo quienes debían responder) | Los tres | Evita porcentajes con denominador equivocado |
| 4 | Convertir códigos a `category`/`Int64`; describir con frecuencias y medianas | Pritchard, Engeln | Evita promediar categorías |
| 4 | Crear un `id` propio en Pritchard; excluir `filter_$` e `iv13r20c2`; normalizar nombres del DANE | Pritchard, DANE | Orden y trazabilidad |
| 5 | Aplicar la encuesta propia y diagnosticarla con este mismo cuaderno | Encuesta | Es la única fuente para moda, maquillaje y jóvenes colombianos |

### 5.7 Síntesis del diagnóstico

Los tres archivos **se pueden cargar y usar**, pero **no están listos para analizar tal como vienen**. Los problemas más serios no son de valores faltantes, sino de **significado de los códigos**: escalas de comparación y satisfacción cuyas etiquetas no corresponden a los valores. La mayoría de los faltantes son **justificados** (saltos del cuestionario o grupo de control); los faltantes ocultos de Pritchard (texto vacío) son la excepción. Con las acciones propuestas, Pritchard y Engeln pueden servir como **antecedentes con datos reales** y el DANE como **contexto**, pero la pregunta analítica, sobre todo la parte de moda y maquillaje en jóvenes colombianos, depende de la encuesta propia.